# 🚦 25 · Hypothesis testing: errors, power, z- and t-tests
### Null and alternative hypotheses, Type I and Type II errors, power and sample size, the three-step recipe (statistic → p-value → decision), the z-test, the t distribution and its table, one-sample, paired and two-sample t-tests — on a cholesterol drug trial and an infection experiment

**BiologyToMath** · Part G — Statistical inference in practice · Module 25 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/25_hypothesis_testing.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, `scipy.integrate` — to check hand-written formulas), pandas, Matplotlib |
| **Prerequisites** | [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) (normal distribution, CDF), [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) (expectation, variance, sample variance), [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) (sampling distribution of the mean, CLT, confidence intervals); [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb) and [Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb) are the estimation side of the same story. |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Hypothesis testing: estimation vs testing as learning strategies; what a test does; null and alternative hypotheses; reject vs retain; the general form $H_0:\theta\in\Theta_0$ vs $H_1:\theta\in\Theta_1$; two-sided and one-sided hypotheses; Type I and Type II errors and the truth × decision table; Type I and Type II error rates, significance level $\alpha$, power $1-\beta$; the two-sided and one-sided error-region pictures; the trade-off between the errors (cut-off, distance $\mu^*-\mu_0$, sample size); performing a test in three steps (test statistic, $p$-value, decision); the $z$-test with known $\sigma$; the $p$-value evidence table; one-sided tests; the $t$ distribution (density, degrees of freedom, heavier tails, limit $\nu\to\infty$) and how to read a $t$ table; $t$-tests when $\sigma$ is unknown; paired $t$-tests |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. **All data in this notebook are SIMULATED by code** and labelled as such: a sample of 100 patients measured after a cholesterol-lowering medication, the paired before/after cholesterol trial of the course toolkit (`simulate_cholesterol_trial`) and the infection experiment (`simulate_tissue_expression`). The cholesterol levels, the population standard deviation and the drug effects are illustrative numbers, not clinical reference values. The $t$ table is generated by code with `scipy.stats.t`.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **State** a biological question as a pair of hypotheses $H_0$ and $H_1$ about a parameter — two-sided or one-sided —
   and **explain** why a test can *reject* or *retain* $H_0$ but never "accept" it.
2. **Define** Type I and Type II errors, their rates $\alpha$ and $\beta$, and the power $1-\beta$, and **fill in** the
   truth × decision table for a concrete experiment.
3. **Draw** and **compute** the error regions of a two-sided and a one-sided test (the two-bell-curve picture), and
   **predict** how $\alpha$, $\beta$ and the power change when you move the cut-off, the true effect or the sample size.
4. **Derive** the sample-size formula $n = \big((z_{1-\alpha/2}+z_{1-\beta})\,\sigma/\Delta\big)^2$ and **use** it to plan
   an experiment.
5. **Perform** a test in three steps — test statistic, $p$-value, decision — for a mean with known $\sigma$ (the
   $z$-test), and **interpret** a $p$-value correctly (and say what it is *not*).
6. **Write down** the density of the $t$ distribution, **explain** degrees of freedom and heavy tails, and **read** a
   $t$ table that you generated yourself.
7. **Carry out** one-sample, paired and two-sample (Welch) $t$-tests by hand and **check** them with `scipy.stats`, and
   **explain** why pairing the cholesterol data increases power so much.
8. **Connect** tests and confidence intervals, and statistical and practical significance.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, simulate 100 patients after medication | 5 min |
| 1 · From estimation to testing | learning strategies, $H_0$ and $H_1$, reject vs retain | 15 min |
| 2 · The form of hypotheses | $\Theta_0$ vs $\Theta_1$, two-sided and one-sided tests | 10 min |
| 3 · Type I and Type II errors | the truth × decision table, $\alpha$, $\beta$, power — by simulation | 15 min |
| 4 · Seeing the errors | recreate the two-sided and one-sided error-region pictures; compute the areas | 20 min |
| 5 · The trade-off | moving the cut-off, the effect, the sample size; power curves; sample-size planning | 25 min |
| 6 · Performing a test: the $z$-test | statistic → $p$-value → decision; one-sided tests; $p$-values under $H_0$; is the coin fair? | 30 min |
| 7 · When $\sigma$ is unknown: the $t$ distribution | density, degrees of freedom, heavy tails, the $t$ table | 25 min |
| 8 · One-sample and paired $t$-tests | the cholesterol trial; pairing and power; tests ↔ confidence intervals | 25 min |
| 9 · Two groups: Welch's $t$-test | one gene, infected vs control — the start of differential expression | 15 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 10 min |

## 🧫 The biological question

A company has developed a new cholesterol-lowering medication. In the population the drug is meant for, total
cholesterol is known to be about **210 mg/dL on average**, with a person-to-person standard deviation of about
**40 mg/dL**. The company gives the drug to **100 patients** and measures their cholesterol afterwards. The average comes
out at a little over 200 mg/dL.

***Did the medication change cholesterol — or would 100 untreated people also, by chance, average 201 mg/dL?***

Every experimental biologist faces this question daily, in many disguises: *does this knockout change the phenotype?
Is this gene expressed differently in infected tissue? Is the allele more common in patients?* The answer is never a
plain yes or no, because biological measurements vary. Hypothesis testing turns "is the difference real?" into a precise
procedure with **known error rates** — and power analysis tells us, *before* we start, how many patients or mice we need
to have a fair chance of seeing an effect.

| Biological question | Mathematical tool | Where |
|---|---|---|
| Did the drug change mean cholesterol? | $H_0:\mu=210$ vs $H_1:\mu\ne210$; the $z$-test | Sections 1, 2, 6 |
| How often would we be fooled — either way? | Type I / Type II error rates, power | Sections 3–4 |
| How many patients do we need? | power curves, the sample-size formula | Section 5 |
| What if we do not know the population SD? | the $t$ distribution, one-sample $t$-test | Sections 7–8 |
| Same patients measured before and after? | the paired $t$-test | Section 8 |
| Is an interferon gene higher in infected spleen? | Welch's two-sample $t$-test | Section 9 |

> 🧪 **Simulated data, with a known truth.** All measurements here are generated by code, so we know what the drug *really*
> does. That lets us check something no real study can: whether each decision of a test was right or wrong.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate, stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `simulate_cholesterol_trial` creates a SIMULATED paired drug trial (LDL cholesterol before and after treatment, with a known true effect); `simulate_tissue_expression` creates the course's SIMULATED infection experiment (spleen, kidney, liver × control/infected × 4 replicates, 400 genes). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools

import numpy as np
import pandas as pd


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}


def simulate_cholesterol_trial(n: int = 12, effect: float = -18.0, seed: int = 2026) -> pd.DataFrame:
    """SIMULATED paired drug trial: LDL cholesterol (mg/dL) of ``n`` patients before and after
    treatment.  Patients differ a lot from each other (between-patient SD 30 mg/dL around 160), the
    measurement/day-to-day noise is smaller (SD 10), and the drug changes LDL by ``effect`` on
    average (the hidden truth).  Columns: patient, before, after."""
    rng = np.random.default_rng(seed)
    level = rng.normal(160.0, 30.0, n)
    before = level + rng.normal(0.0, 10.0, n)
    after = level + effect + rng.normal(0.0, 10.0, n)
    return pd.DataFrame({"patient": [f"P{i + 1:02d}" for i in range(n)],
                         "before": before.round(1), "after": after.round(1)})

print("🧰 toolkit loaded:", ", ".join(sorted(['HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', 'simulate_cholesterol_trial', 'simulate_tissue_expression'])))

In [ ]:
# ── 0.1 The study of Sections 1–6: 100 patients measured after the medication (SIMULATED) ──
MU0 = 210.0          # mg/dL — the known average cholesterol of the untreated population (the null value)
SIGMA = 40.0         # mg/dL — the known person-to-person standard deviation
N_PATIENTS = 100
TRUE_MU = 200.0      # the hidden truth of the simulation: the drug lowers the average by 10 mg/dL

rng_trial = np.random.default_rng(SEED + 6)
cholesterol = np.round(rng_trial.normal(TRUE_MU, SIGMA, N_PATIENTS), 0)   # one value per patient, mg/dL

print(f"n = {cholesterol.size} patients; first ten values (mg/dL): {cholesterol[:10]}")
print(f"sample mean = {cholesterol.mean():.2f} mg/dL   sample SD = {cholesterol.std(ddof=1):.2f} mg/dL")
print(f"null value mu0 = {MU0:.0f} mg/dL; difference = {cholesterol.mean() - MU0:+.2f} mg/dL")

🔍 **What just happened?** We drew 100 cholesterol values from a normal distribution whose true mean is 200 mg/dL — the
drug really works in this simulation, lowering the average by 10 mg/dL. The sample average is **201.07 mg/dL**, about
9 mg/dL below the untreated average of 210, and the sample standard deviation (38.11 mg/dL) is close to the true 40.
A real study would see only these 100 numbers, not `TRUE_MU`. The whole notebook is about one question: *is a difference
of −8.9 mg/dL large enough to be convincing, given how much individual patients vary?*

---
## 1 · From estimation to testing

> 📍 **Where we are.** In Modules 23 and 24 we *estimated* parameters: given $X_1,\dots,X_n\sim\mathcal N(\mu,\sigma^2)$,
> maximum likelihood says $\hat\mu = \bar X_n$, and a Bayesian analysis gives a whole posterior for $\mu$. Module 22 put
> an error bar on the estimate (a confidence interval). Now the question changes: we have a **specific claim** about the
> parameter — "the drug does nothing, $\mu = 210$" — and we want to know whether the data are compatible with it.

**Two learning strategies for the same data.** Consider $n$ samples from a normal distribution,
$X_1,\dots,X_n\sim\mathcal N(\mu,\sigma^2)$.

* **Estimation** (maximum likelihood, Bayesian estimation): *what is $\mu$?* The answer is a number (or a distribution).
* **Hypothesis testing** is used when you can form a **hypothesis that you want to test using data**: *is $\mu$ different
  from 210?* The answer is a decision — reject or retain the hypothesis — together with a number that measures the
  evidence.

**What is hypothesis testing?** A technique to evaluate whether a model fitted to the data matches our assumptions about
the data, which assigns a **numerical value** — most often the ***p*-value** — to how well they match.

**The study, in the language of models.**

* *Assume* the cholesterol levels after medication follow a normal distribution $\mathcal N(\mu,\sigma^2)$.
* *Hypothesis:* the known cholesterol level of untreated people is $\mu_0 = 210$ mg/dL, with known $\sigma$ = 40 mg/dL.
  With medication, did the level change?
* *Data:* cholesterol measured after medication in $n = 100$ people.

⚠️ **Units.** Blood cholesterol is a *concentration*: milligrams per decilitre of blood (mg/dL; 210 mg/dL ≈ 5.4 mmol/L).
It is not a volume in millilitres.

In [ ]:
# ── 1.1 Look at the data before testing anything (SIMULATED) ──
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(cholesterol, bins=18, color="tab:blue", alpha=0.6, edgecolor="white", label="100 patients after medication")
ax.axvline(MU0, color="tab:orange", lw=2, label=f"untreated average μ0 = {MU0:.0f} mg/dL")
ax.axvline(cholesterol.mean(), color="k", ls="--", lw=1.5, label=f"sample mean = {cholesterol.mean():.1f} mg/dL")
ax.set_xlabel("total cholesterol (mg/dL)")
ax.set_ylabel("number of patients")
ax.set_title("cholesterol after medication (SIMULATED)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
print(f"patients below 210 mg/dL: {np.sum(cholesterol < MU0)} of {cholesterol.size}; "
      f"range {cholesterol.min():.0f}–{cholesterol.max():.0f} mg/dL")

🔍 **What just happened?** The histogram is wide: individual patients range from 98 to 280 mg/dL. The sample mean
(dashed) sits a little to the left of the untreated average 210 (orange), and 57 of the 100 patients are below 210. A shift
of 9 mg/dL is small compared with the 40 mg/dL spread *between people* — so the question "real or chance?" is genuinely
open. The point of a test is to make this judgement with a rule whose error rate we know.

### The two hypotheses

* **Null hypothesis $H_0$** — the assumption that there is **no effect or no difference**. Example: *the drug does not
  change cholesterol levels*; $H_0: \mu = 210$.
* **Alternative hypothesis $H_1$** — the assumption that there **is an effect or a difference**. Example: *the drug
  **does** change cholesterol levels*; $H_1: \mu \ne 210$.

Another way to say the same thing: $H_1$ is the claim we are asking about (the event whose evidence we want to measure),
and $H_0$ collects **all the other possibilities**. Together they cover every possible value of the parameter.

**The outcome of a test** is one of two decisions:

* **Reject $H_0$** → the evidence supports $H_1$ (*"the cholesterol level changed"*);
* **Retain $H_0$** → there is not enough evidence to support $H_1$ (*"we cannot tell whether it changed"*).

Choosing $H_0$ means "retain $H_0$"; choosing $H_1$ means "reject $H_0$".

⚠️ **Common confusion — "retain" is not "accept".** Retaining $H_0$ does **not** show that the drug has no effect. It
says only that *these* data are compatible with no effect — perhaps because the study was too small to see a real effect
(Section 5). A courtroom analogy helps: the defendant is presumed innocent ($H_0$); the verdict is "guilty" (reject $H_0$)
or "not guilty" (retain $H_0$) — and "not guilty" is not the same as "proven innocent". *Absence of evidence is not
evidence of absence.*

✏️ **Your turn.** Write $H_0$ and $H_1$ in words *and* in symbols for: (a) a knockout mouse line — does knocking out gene
*X* change mean body weight (wild-type mean 25 g)? (b) a seed company claims its new maize variety yields **more** than
the standard 9 t/ha. Which of the two is naturally one-sided?

---
## 2 · The form of hypotheses

> 📍 **Where we are.** We have the words; now the mathematics. A hypothesis is a statement about a **parameter** $\theta$ of
> the model (here $\theta = \mu$), and the two hypotheses split the set of possible parameter values into two pieces — sets
> and partitions from Module 03.

**General form.** Let $\Theta$ be the set of all possible values of the parameter $\theta$ (the *parameter space*; for a
mean, $\Theta = \mathbb R$). To test a hypothesis about $\theta\in\Theta$, divide $\Theta$ into two regions:
$\Theta_0$ for $H_0$ and $\Theta_1$ for $H_1$, with

$$\Theta_0 \cup \Theta_1 = \Theta, \qquad \Theta_0\cap\Theta_1 = \varnothing, \qquad\text{and then}\qquad H_0:\ \theta\in\Theta_0, \qquad H_1:\ \theta\in\Theta_1 .$$

*In words:* every possible value of the parameter belongs to exactly one of the two hypotheses — $\{\Theta_0,\Theta_1\}$ is
a partition of $\Theta$. A hypothesis whose set contains a single value ($\Theta_0 = \{210\}$) is called **simple**; one
whose set contains many values ($\Theta_1 = \{\mu : \mu\ne 210\}$) is called **composite**.

**The three standard forms** (with a known reference value $\theta_0$):

| Test | $H_0$ | $H_1$ | Cholesterol example |
|---|---|---|---|
| two-sided | $\theta = \theta_0$ | $\theta \ne \theta_0$ | "does the drug *change* cholesterol?" |
| one-sided (left) | $\theta \ge \theta_0$ | $\theta < \theta_0$ | "does the drug *lower* cholesterol?" |
| one-sided (right) | $\theta \le \theta_0$ | $\theta > \theta_0$ | "does a high-fat diet *raise* cholesterol?" |

*In words:* a **two-sided** test looks for a change in either direction; a **one-sided** test looks in one direction only
and lumps "no change" together with "a change in the other direction" into $H_0$.

⚠️ **Choose the form before you look at the data.** Picking the one-sided direction *after* seeing which way the data
point halves the $p$-value (Section 6) and silently doubles the false-positive rate.

In [ ]:
# ── 2.1 The three forms as partitions of the parameter line Θ = ℝ ──
fig, axes = plt.subplots(3, 1, figsize=(7.5, 3.6), sharex=True)
lo, hi, theta0 = 170, 250, MU0
forms = [("two-sided:  H0: μ = 210   vs   H1: μ ≠ 210", "two"),
         ("one-sided:  H0: μ ≥ 210   vs   H1: μ < 210", "left"),
         ("one-sided:  H0: μ ≤ 210   vs   H1: μ > 210", "right")]
for ax, (title, kind) in zip(axes, forms):
    ax.axhline(0, color="0.5", lw=1)
    if kind == "two":
        ax.plot([lo, hi], [0, 0], color="tab:blue", lw=6, solid_capstyle="butt", label="Θ1 (H1)")
        ax.plot([theta0], [0], "o", color="tab:orange", ms=10, label="Θ0 (H0)", zorder=3)
    elif kind == "left":
        ax.plot([lo, theta0], [0, 0], color="tab:blue", lw=6, solid_capstyle="butt", label="Θ1 (H1)")
        ax.plot([theta0, hi], [0, 0], color="tab:orange", lw=6, solid_capstyle="butt", label="Θ0 (H0)")
    else:
        ax.plot([lo, theta0], [0, 0], color="tab:orange", lw=6, solid_capstyle="butt", label="Θ0 (H0)")
        ax.plot([theta0, hi], [0, 0], color="tab:blue", lw=6, solid_capstyle="butt", label="Θ1 (H1)")
    ax.axvline(theta0, color="k", ls=":", lw=1)
    ax.set_yticks([])
    ax.set_title(title, loc="left", fontsize=9)
    ax.spines["left"].set_visible(False)
    ax.legend(loc="center right", fontsize=7, ncol=2, frameon=False, bbox_to_anchor=(1.0, 1.25))
axes[-1].set_xlabel("possible values of the true mean μ (mg/dL)")
plt.tight_layout()
plt.show()

🔍 **What just happened?** Each line is the parameter space $\Theta=\mathbb R$ (zoomed to 170–250 mg/dL), coloured by the
hypothesis each value belongs to. In the two-sided test $\Theta_0$ is a single point (a *simple* null) and $\Theta_1$ is
everything else; in the one-sided tests both pieces are half-lines (*composite*), meeting at $\theta_0 = 210$. In every
picture the two colours together cover the whole line and never overlap — a partition.

### 📘 Lecture notes — hypotheses and decisions

- **Estimation vs testing.** For data $X_1,\dots,X_n\sim\mathcal N(\mu,\sigma^2)$, maximum-likelihood and Bayesian
  estimation ask *what is $\mu$?*; **hypothesis testing** is used when you can form a hypothesis that you want to test
  using data. It evaluates whether a fitted model matches our assumptions and assigns a number (the $p$-value) to the
  match.
- **Null hypothesis $H_0$:** no effect / no difference ("the drug does not change cholesterol", $\mu = 210$).
  **Alternative $H_1$:** an effect / a difference ("the drug does change cholesterol", $\mu\ne210$).
- **Decisions:** *reject $H_0$* (evidence supports $H_1$) or *retain $H_0$* (not enough evidence for $H_1$). Never
  "accept $H_0$": absence of evidence is not evidence of absence.
- **General form:** split the parameter space $\Theta$ into $\Theta_0$ and $\Theta_1$ with $\Theta_0\cup\Theta_1 = \Theta$
  (and no overlap); $H_0:\theta\in\Theta_0$, $H_1:\theta\in\Theta_1$. Simple hypothesis = one value; composite = many.
- **Two-sided:** $H_0:\theta=\theta_0$ vs $H_1:\theta\ne\theta_0$. **One-sided:** $H_0:\theta\ge\theta_0$ vs
  $H_1:\theta<\theta_0$, or $H_0:\theta\le\theta_0$ vs $H_1:\theta>\theta_0$.
- **Worked example:** known cholesterol level 210 mg/dL with known $\sigma$; with medication, did it change?
  $H_0:\mu=210$, $H_1:\mu\ne210$.

> 🎓 **Exam tip.** Hypotheses are always about the **parameter** ($\mu$), never about the data ($\bar x$): "$H_0: \bar x = 210$" is wrong. And state the direction of a one-sided test *before* looking at the data.

✏️ **Your turn.** For a genotype × disease question — "is the risk allele more frequent in patients than in controls?" —
write $\Theta$, $\Theta_0$ and $\Theta_1$ for the parameter $\theta = p_{\text{patients}} - p_{\text{controls}}$. Is $H_0$
simple or composite?

---
## 3 · Type I and Type II errors

> 📍 **Where we are.** A test makes a decision from noisy data, so it can be wrong. There are exactly two ways to be wrong,
> and they have very different consequences in biology and medicine.

**Type I error (false positive).** *Rejecting the null hypothesis $H_0$ when $H_0$ is actually true.*
Example: the test says cholesterol changed — but in reality it did not.

**Type II error (false negative).** *Retaining the null hypothesis $H_0$ when $H_0$ is actually false.*
Example: the test says cholesterol did not change — but in reality it did.

Reality has two branches ($H_0$ true or $H_0$ false) and so does our decision; crossing them gives the **truth × decision
table**:

| Truth ↓ / Decision → | Retain $H_0$ | Reject $H_0$ |
|---|---|---|
| $H_0$ true | ✅ correct | ❌ **Type I error** (rate $\alpha$) |
| $H_1$ true | ❌ **Type II error** (rate $\beta$) | ✅ correct (rate $1-\beta$ = **power**) |

**Error rates.** Each row is a *conditional* probability — conditioned on which hypothesis is true (Module 17):

$$\alpha = P(\text{reject } H_0 \mid H_0 \text{ is true}), \qquad \beta = P(\text{retain } H_0 \mid H_1 \text{ is true}), \qquad \text{power} = 1-\beta = P(\text{reject } H_0\mid H_1\text{ is true}).$$

*In words:* $\alpha$ is how often the test cries wolf when there is no wolf; $\beta$ is how often it stays silent when
there is one; the power is how often it catches a real effect. Higher-power tests have a stronger ability to detect
signals for $H_1$. The Type I error rate we are willing to tolerate is called the **significance level** of the test and
is also written $\alpha$ (typically 0.05).

⚠️ **Common confusion — joint vs conditional.** You may see the Type I error rate written $p(\text{reject }H_0,\ H_0\text{
is true})$ with a comma. Read it as the *conditional* probability above: $\alpha$ is computed in a world where $H_0$ holds,
and it does not depend on how often $H_0$ is true in general. The joint probability $P(\text{reject}\cap H_0)$ is a
different number ($=\alpha\,P(H_0)$ by the product rule) — we use it in the 🌉 Bridge of Section 6.

**The rule we will study.** Under $H_0$ the sample mean of 100 patients is approximately $\mathcal N(210,\ 40^2/100)$, i.e.
centred on 210 with standard deviation (the *standard error*, Module 22) $40/\sqrt{100} = 4$ mg/dL. A natural rule:

$$\text{reject } H_0 \iff \left|\frac{\bar x - 210}{4}\right| \ge 1.96 .$$

*In words:* reject when the sample mean is more than 1.96 standard errors away from 210. Section 6 explains where 1.96
comes from; here we simply *measure* how often this rule errs, by repeating the whole study thousands of times — in one
world where the drug does nothing ($\mu=210$) and in one where it lowers cholesterol by 10 ($\mu=200$).

In [ ]:
# ── 3.1 Repeat the study 20,000 times in each world and count the decisions (SIMULATED) ──
def reject_rule(samples, mu0=MU0, sigma=SIGMA, cutoff=1.96):
    """Decision of the two-sided rule |x̄ − mu0| / (sigma/√n) ≥ cutoff, applied to every row of `samples`."""
    n = samples.shape[1]
    z = (samples.mean(axis=1) - mu0) / (sigma / np.sqrt(n))
    return np.abs(z) >= cutoff


rng_err = np.random.default_rng(SEED + 31)
n_rep = 20_000
world_h0 = rng_err.normal(MU0, SIGMA, (n_rep, N_PATIENTS))     # drug does nothing: μ = 210
world_h1 = rng_err.normal(TRUE_MU, SIGMA, (n_rep, N_PATIENTS)) # drug works: μ = 200
rej_h0, rej_h1 = reject_rule(world_h0), reject_rule(world_h1)

table = pd.DataFrame({"retain H0": [1 - rej_h0.mean(), 1 - rej_h1.mean()],
                      "reject H0": [rej_h0.mean(), rej_h1.mean()]},
                     index=["H0 true (μ = 210)", "H1 true (μ = 200)"])
print("fraction of the 20,000 simulated studies in each cell (each row sums to 1):")
print(table.round(4))
print(f"\nType I error rate α ≈ {rej_h0.mean():.4f}   Type II error rate β ≈ {1 - rej_h1.mean():.4f}   "
      f"power ≈ {rej_h1.mean():.4f}")

🔍 **What just happened?** In the world where the drug does nothing, the rule wrongly rejected $H_0$ in about **5 %** of
the 20,000 studies — the Type I error rate $\alpha\approx 0.05$ that the cut-off 1.96 was designed to give. In the world
where the drug lowers cholesterol by 10 mg/dL, it correctly rejected $H_0$ in about **70 %** of the studies (the power),
and missed the effect in the remaining 30 % (Type II errors, $\beta\approx0.30$). Each *row* sums to 1, because each row
is a conditional distribution of the decision given the truth. Notice the asymmetry: we *chose* $\alpha$, but $\beta$ was
whatever the sample size and effect size gave us. A study with power 0.7 misses a real 10 mg/dL effect almost one time in
three.

**Which error is worse?** It depends on the biology. In drug development, a Type I error can send an ineffective drug into
expensive trials (or onto the market); a Type II error can kill a useful drug forever. In a screening test for a serious
disease, a false negative sends a sick person home. Fixing $\alpha$ at a small value and then designing the study for
high power (Section 5) is the standard compromise.

### 📘 Lecture notes — Type I and Type II errors, power

- **Type I error (false positive):** rejecting $H_0$ when $H_0$ is true (test says cholesterol changed; in reality it did
  not). **Type II error (false negative):** retaining $H_0$ when $H_0$ is false (test says no change; in reality it changed).
- **Table:** truth ($H_0$ / $H_1$) × decision (retain / reject): correct, Type I / Type II, correct.
- **Rates:** Type I error rate $\alpha = P(\text{reject } H_0\mid H_0\text{ true})$; Type II error rate
  $\beta = P(\text{retain } H_0\mid H_1\text{ true})$; **power** $= 1-\beta$ — higher power means a stronger ability to
  detect signals for $H_1$.
- **Significance level** = the Type I error rate we accept, chosen in advance (0.01, 0.05, 0.1 are common).
- **Worked numbers** (our rule $|\bar x-210|/4\ge1.96$, $n=100$, $\sigma=40$): $\alpha\approx0.05$; if the true mean is
  200, power $\approx0.70$, $\beta\approx0.30$.

> 🎓 **Exam tip.** "Type I = false positive = $\alpha$ = rejecting a true $H_0$"; "Type II = false negative = $\beta$". Power is $1-\beta$, **not** $1-\alpha$ ($1-\alpha$ is the probability of correctly *retaining* a true $H_0$).

✏️ **Your turn.** Change `TRUE_MU` in the simulation of `world_h1` to 205 and then to 190 (keep everything else). How
do the power and the Type I error rate change? Why does the Type I error rate not change at all?

---
## 4 · Seeing the errors: the two-bell-curve picture

> 📍 **Where we are.** Section 3 *counted* errors by brute force. Now we draw them, and compute them exactly from the
> normal CDF of Module 18. The key picture of this module shows two bell curves: the distribution of the test statistic
> if $H_0$ is true, and if $H_1$ is true.

**Notation.** $\mu_0$ is the location of the null hypothesis $H_0$ (**known** — 210 mg/dL). $\mu^*$ is the location of the
true alternative (usually **unknown** — it is what the drug really does). Both curves describe where the statistic
$\omega$ (for example the sample mean) is likely to land. We choose **cut-offs** (green lines): outside them we choose
$H_1$ (reject), between them we retain $H_0$.

* **Type I error** = the area of the **$H_0$** curve in the reject region (blue).
* **Type II error** = the area of the **$H_1$** curve in the retain region (red).
* **Power** = the area of the $H_1$ curve in the reject region (green hatching).

🧮 **Math refresher — areas under a normal curve.** For $W\sim\mathcal N(m, s^2)$, the CDF (Module 18)
$P(W\le c) = \Phi\big(\frac{c-m}{s}\big)$, where $\Phi$ is the standard normal CDF (`stats.norm.cdf`). So
$P(a\le W\le b) = \Phi\big(\frac{b-m}{s}\big)-\Phi\big(\frac{a-m}{s}\big)$ and $P(W\ge c) = 1-\Phi\big(\frac{c-m}{s}\big)$.
*In words:* standardise the cut-off (how many SDs from the centre is it?) and read the area from $\Phi$.

We first draw the picture in **standard units** (the statistic has SD 1 under both hypotheses), with $\mu_0 = 0$,
$\mu^* = 3$ and cut-offs at $\pm1$, exactly as in the classic textbook figure.

In [ ]:
# ── 4.1 The two-sided picture: H0: μ = μ0 vs H1: μ ≠ μ0 (standard units) ──
def shade(ax, x, y, mask, **kw):
    """Fill the area under the curve y(x) where mask is True."""
    ax.fill_between(x, 0, y, where=mask, interpolate=True, **kw)


mu0_s, mu_star_s, c = 0.0, 3.0, 1.0                # the textbook picture: cut-offs at ±1
w = np.linspace(-3.5, 6.5, 1001)
f0, f1 = stats.norm.pdf(w, mu0_s, 1), stats.norm.pdf(w, mu_star_s, 1)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(w, f0, color="tab:blue", label="H0: distribution of ω if μ = μ0")
ax.plot(w, f1, color="tab:red", label="H1: distribution of ω if μ = μ*")
shade(ax, w, f0, np.abs(w - mu0_s) >= c, color="tab:blue", alpha=0.3, label="Type I error (α)")
shade(ax, w, f1, np.abs(w - mu0_s) < c, color="tab:red", alpha=0.35, label="Type II error (β)")
ax.vlines([mu0_s - c, mu0_s + c], 0, 0.27, color="tab:green", lw=3)
for m in (mu0_s, mu_star_s):
    ax.axvline(m, color="k", ls="--", lw=0.8)
ax.set_xticks([mu0_s, mu_star_s]); ax.set_xticklabels(["μ0", "μ*"])
ax.set_xlabel("ω  (value of the test statistic)"); ax.set_ylabel("p(ω)")
for x_txt, txt in [(-2.4, "choose H1"), (0, "retain H0"), (2.0, "choose H1")]:
    ax.text(x_txt, -0.075, txt, ha="center", fontsize=8, color="tab:green")
ax.set_ylim(-0.09, 0.45); ax.set_yticks([0, 0.1, 0.2, 0.3, 0.4]); ax.axhline(0, color="0.6", lw=0.6)
ax.legend(loc="upper right", fontsize=7)
ax.set_title("two-sided test: cut-offs at μ0 ± 1 (green lines)")
plt.tight_layout(); plt.show()

alpha_2s = 2 * stats.norm.sf(c)                                       # both tails of H0
beta_2s = stats.norm.cdf(c, mu_star_s) - stats.norm.cdf(-c, mu_star_s)  # H1 between the cut-offs
print(f"Type I error α = P(|ω| ≥ 1 | H0)       = 2·(1 − Φ(1))         = {alpha_2s:.4f}")
print(f"Type II error β = P(−1 < ω < 1 | H1)  = Φ(1 − 3) − Φ(−1 − 3) = {beta_2s:.4f}")
print(f"power = 1 − β = {1 - beta_2s:.4f}")

🔍 **What just happened?** We redrew the classic figure. The blue bell is the null distribution, the red bell the
alternative. The blue tails beyond the green lines (both sides!) are the Type I error: **α = 0.317** — with cut-offs only
one standard deviation from $\mu_0$, a true null would be rejected almost one time in three. The thin red sliver between
the green lines is the Type II error: **β = 0.0227**, so the power is 0.977. These cut-offs are far too lenient for
practice (the usual two-sided cut-offs are ±1.96, giving α = 0.05), but they make both areas visible. Below the axis, the
green labels show the decision regions: *choose $H_1$ | retain $H_0$ | choose $H_1$*.

In formulas, for a statistic with null distribution $\mathcal N(\mu_0, s^2)$ and alternative distribution
$\mathcal N(\mu^*, s^2)$, and cut-offs $c_{lo} < c_{hi}$:

$$\alpha = \Phi\Big(\frac{c_{lo}-\mu_0}{s}\Big) + 1 - \Phi\Big(\frac{c_{hi}-\mu_0}{s}\Big),\qquad
\beta = \Phi\Big(\frac{c_{hi}-\mu^*}{s}\Big) - \Phi\Big(\frac{c_{lo}-\mu^*}{s}\Big).$$

*In words:* $\alpha$ is the null curve's area outside the cut-offs; $\beta$ is the alternative curve's area between them.

**The one-sided picture.** For $H_0:\mu\le\mu_0$ vs $H_1:\mu>\mu_0$ there is a single cut-off on the right; only the right
tail of $H_0$ is a Type I error, and the power is the part of the $H_1$ curve to the right of the cut-off.

In [ ]:
# ── 4.2 The one-sided picture: H0: μ ≤ μ0 vs H1: μ > μ0, with the power hatched ──
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(w, f0, color="tab:blue", label="H0 (μ = μ0)")
ax.plot(w, f1, color="tab:red", label="H1 (μ = μ*)")
shade(ax, w, f0, w >= c, color="tab:blue", alpha=0.3, label="Type I error (α)")
shade(ax, w, f1, w < c, color="tab:red", alpha=0.35, label="Type II error (β)")
ax.fill_between(w, 0, f1, where=w >= c, facecolor="none", edgecolor="tab:green", hatch="////", lw=0,
                label="power (1 − β)")
f1_near = stats.norm.pdf(w, 2.0, 1)                                  # a second, smaller true effect
ax.plot(w, f1_near, color="tab:purple", ls=":", label="another H1: μ* = 2 (smaller effect)")
ax.vlines(c, 0, 0.27, color="tab:green", lw=3)
for m in (mu0_s, mu_star_s):
    ax.axvline(m, color="k", ls="--", lw=0.8)
ax.set_xticks([mu0_s, 2.0, mu_star_s]); ax.set_xticklabels(["μ0", "2", "μ*"])
ax.set_xlabel("ω  (value of the test statistic)"); ax.set_ylabel("p(ω)")
ax.text(-1.2, -0.075, "retain H0", ha="center", fontsize=8, color="tab:green")
ax.text(3.0, -0.075, "choose H1", ha="center", fontsize=8, color="tab:green")
ax.set_ylim(-0.09, 0.45); ax.set_yticks([0, 0.1, 0.2, 0.3, 0.4]); ax.axhline(0, color="0.6", lw=0.6)
ax.legend(loc="upper left", fontsize=7)
ax.set_title("one-sided test: a single cut-off at μ0 + 1")
plt.tight_layout(); plt.show()

print(f"α = P(ω ≥ 1 | H0) = 1 − Φ(1) = {stats.norm.sf(c):.4f}")
print(f"β = P(ω < 1 | μ* = 3) = Φ(1 − 3) = {stats.norm.cdf(c - 3):.4f}   power = {stats.norm.sf(c - 3):.4f}")
print(f"if instead μ* = 2:  β = Φ(1 − 2) = {stats.norm.cdf(c - 2):.4f}   power = {stats.norm.sf(c - 2):.4f}")

🔍 **What just happened?** With one cut-off at $\mu_0+1$, only the *right* tail of $H_0$ counts as a Type I error, so
**α = 0.1587** — half of the two-sided value with the same cut-off. The power (green hatching) is **0.9772** when
$\mu^*=3$. The dotted purple curve shows what happens if the true effect is smaller ($\mu^*=2$): more of it falls left of the
cut-off, so β grows to 0.1587 and the power drops to 0.8413. The farther the truth is from $\mu_0$, the easier it is to
detect — the next section makes this precise.

Let us check the formula with a simulation in **cholesterol units**: the sample mean of 100 patients has standard error
$s = 40/\sqrt{100} = 4$ mg/dL; the usual two-sided cut-offs are $210\pm1.96\cdot4 = [202.16,\ 217.84]$, and the drug's
true mean is $\mu^* = 200$.

In [ ]:
# ── 4.3 Error rates from the formula vs from simulation (cholesterol units) ──
def error_rates(c_lo, c_hi, mu0, mu_star, se):
    """α, β and power of the rule 'reject unless c_lo < x̄ < c_hi' when x̄ ~ N(mu0, se²) under H0
    and x̄ ~ N(mu_star, se²) under H1."""
    alpha = stats.norm.cdf((c_lo - mu0) / se) + stats.norm.sf((c_hi - mu0) / se)
    beta = stats.norm.cdf((c_hi - mu_star) / se) - stats.norm.cdf((c_lo - mu_star) / se)
    return alpha, beta, 1 - beta


se = SIGMA / np.sqrt(N_PATIENTS)
c_lo, c_hi = MU0 - 1.96 * se, MU0 + 1.96 * se
a_f, b_f, pw_f = error_rates(c_lo, c_hi, MU0, TRUE_MU, se)
xbar_h0, xbar_h1 = world_h0.mean(axis=1), world_h1.mean(axis=1)     # the 20,000 studies of Section 3
a_s = np.mean((xbar_h0 <= c_lo) | (xbar_h0 >= c_hi))
b_s = np.mean((xbar_h1 > c_lo) & (xbar_h1 < c_hi))
print(f"standard error = {se:.1f} mg/dL; retain H0 if {c_lo:.2f} < x̄ < {c_hi:.2f} mg/dL")
print(f"           formula    simulation (20,000 studies)")
print(f"alpha      {a_f:.4f}     {a_s:.4f}")
print(f"beta       {b_f:.4f}     {b_s:.4f}")
print(f"power      {pw_f:.4f}     {1 - b_s:.4f}")

🔍 **What just happened?** The two-line formula and the brute-force count of Section 3 agree to two or three decimals:
α = 0.0500 by formula, β = 0.2946 and power **0.7054**. So with 100 patients, $\sigma=40$ and a true drug effect of
−10 mg/dL, this study has about a 70 % chance of detecting the effect. (The simulation differs from the formula only by
Monte-Carlo noise of about $\pm0.003$.) We can now explore the trade-offs without simulating at all.

✏️ **Your turn.** Redraw the two-sided picture of 4.1 with the realistic cut-offs `c = 1.96`. Which area becomes tiny,
and which becomes large? Write down α and β before you run the cell.

---
## 5 · The trade-off: no free lunch

> 📍 **Where we are.** We can compute α, β and power for any cut-off. Now we ask the questions an experimenter must
> answer before collecting data: *how strict should the cut-off be, and how many samples do I need?*

**Questions to think about first** (answer them before running the next cells):

1. If $\alpha$ increases (the cut-off moves towards $\mu_0$), what happens to the Type II error $\beta$? To the power $1-\beta$?
2. If $\mu^*$ is far away from $\mu_0$, what happens to $\beta$ and to the power? Is the signal for $H_1$ in the data easier or
   harder to detect?

We use the one-sided test $H_0:\mu\le\mu_0$ vs $H_1:\mu>\mu_0$ in standard units, with the true alternative at
$\mu^* = 2.5$, and move the cut-off through four positions (a hand-operated "slider").

In [ ]:
# ── 5.1 Moving the cut-off: four snapshots of the trade-off (one-sided, standard units) ──
mu_star_t = 2.5
cutoffs = [stats.norm.isf(a) for a in (0.20, 0.10, 0.05, 0.01)]     # cut-offs giving α = 0.20, 0.10, 0.05, 0.01
fig, axes = plt.subplots(1, 4, figsize=(12, 2.9), sharey=True)
rows = []
for ax, cut in zip(axes, cutoffs):
    g0, g1 = stats.norm.pdf(w, 0, 1), stats.norm.pdf(w, mu_star_t, 1)
    ax.plot(w, g0, color="tab:blue"); ax.plot(w, g1, color="tab:red")
    shade(ax, w, g0, w >= cut, color="tab:blue", alpha=0.35)
    shade(ax, w, g1, w < cut, color="tab:red", alpha=0.35)
    ax.axvline(cut, ymax=0.7, color="tab:green", lw=2.5)
    a, b = stats.norm.sf(cut), stats.norm.cdf(cut - mu_star_t)
    rows.append({"cut-off": cut, "alpha": a, "beta": b, "power": 1 - b})
    ax.set_title(f"cut-off {cut:.2f}:  α = {a:.2f},  β = {b:.2f}", fontsize=9)
    ax.set_xticks([0, mu_star_t]); ax.set_xticklabels(["μ0", "μ*"]); ax.set_xlabel("ω")
axes[0].set_ylabel("p(ω)")
fig.suptitle("moving the cut-off to the right: α (blue) shrinks, β (red) grows", fontsize=10)
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).round(4).to_string(index=False))

🔍 **What just happened?** As the cut-off moves right from 0.84 to 2.33, the blue area shrinks from α = 0.20 to 0.01 —
but the red area grows from β = 0.05 to 0.43, so the power falls from 0.95 to 0.57. **Answer to question 1:** if α
increases, β *decreases* and the power *increases*; if α decreases, β increases and the power decreases. With the
same data, you cannot make both errors small at once by moving the cut-off — there is no free lunch.

Now keep the cut-off fixed (α = 0.05) and move the truth.

In [ ]:
# ── 5.2 Moving the truth μ*: a fixed cut-off, four effect sizes ──
cut05 = stats.norm.isf(0.05)                                         # one-sided α = 0.05 → cut-off 1.645
fig, axes = plt.subplots(1, 4, figsize=(12, 2.9), sharey=True)
for ax, ms in zip(axes, [0.5, 1.5, 2.5, 3.5]):
    g0, g1 = stats.norm.pdf(w, 0, 1), stats.norm.pdf(w, ms, 1)
    ax.plot(w, g0, color="tab:blue"); ax.plot(w, g1, color="tab:red")
    shade(ax, w, g0, w >= cut05, color="tab:blue", alpha=0.35)
    shade(ax, w, g1, w < cut05, color="tab:red", alpha=0.35)
    ax.axvline(cut05, ymax=0.7, color="tab:green", lw=2.5)
    ax.set_title(f"μ* − μ0 = {ms}:  α = {stats.norm.sf(cut05):.2f},  β = {stats.norm.cdf(cut05 - ms):.2f}", fontsize=9)
    ax.set_xlabel("ω")
axes[0].set_ylabel("p(ω)")
fig.suptitle("moving μ* away from μ0 at a fixed cut-off: α stays 0.05, β shrinks", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the cut-off fixed at 1.645, the blue area is **0.05 in every panel** — α depends only on
the null distribution and the cut-off, not on the truth. The red area shrinks from β = 0.87 (true effect 0.5 SE: almost
undetectable) to β = 0.03 (effect 3.5 SE). **Answer to question 2:** as $\mu^*$ moves away from $\mu_0$, β goes down,
the power goes up, and the signal for $H_1$ becomes *easier* to detect.

⚠️ **Common confusion — "as $\mu^*$ and $\mu_0$ move apart, both errors go down."** Only β goes down: α is fixed by the
cut-off you chose. What *is* true is that a larger separation lets you afford a stricter cut-off (smaller α) for the same
β — so the attainable *pair* (α, β) improves. The curves below show exactly this.

**What sets the separation?** In real units the two bell curves are sampling distributions of $\bar X_n$, with SD
$\sigma/\sqrt n$. Measured in standard errors, the separation is

$$\frac{\mu^*-\mu_0}{\sigma/\sqrt n} \;=\; \sqrt n\,\frac{\Delta}{\sigma},\qquad \Delta = \mu^*-\mu_0 .$$

*In words:* the effect $\Delta$ counts in units of the biological spread $\sigma$, and the evidence grows with the square
root of the sample size: four times as many patients doubles the separation. This is the experimenter's real lever.

In [ ]:
# ── 5.3 The whole trade-off at once: β against α, and power against the true effect ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
alphas = np.linspace(0.001, 0.5, 300)
for sep in [1.0, 2.0, 3.0]:
    betas = stats.norm.cdf(stats.norm.isf(alphas) - sep)                # one-sided test
    axes[0].plot(alphas, betas, label=f"separation √n·Δ/σ = {sep:.0f}")
axes[0].axvline(0.05, color="0.5", ls=":", lw=1)
axes[0].set_xlabel("Type I error rate α"); axes[0].set_ylabel("Type II error rate β")
axes[0].set_title("each curve = all (α, β) pairs one design can reach"); axes[0].legend()

delta = np.linspace(-25, 25, 401)                                       # true effect μ* − 210 (mg/dL)
for n in [25, 100, 400]:
    se_n = SIGMA / np.sqrt(n)
    z_crit = stats.norm.isf(0.025)
    power = stats.norm.sf(z_crit - delta / se_n) + stats.norm.cdf(-z_crit - delta / se_n)   # two-sided
    axes[1].plot(delta, power, label=f"n = {n}")
axes[1].axhline(0.05, color="tab:blue", ls=":", lw=1); axes[1].axhline(0.8, color="0.5", ls="--", lw=1)
axes[1].set_xlabel("true effect μ* − 210 (mg/dL)"); axes[1].set_ylabel("power = P(reject H0)")
axes[1].set_title("power curves of the two-sided z-test (α = 0.05, σ = 40)"); axes[1].legend()
plt.tight_layout(); plt.show()

for n in [25, 100, 400]:
    se_n = SIGMA / np.sqrt(n)
    pw = stats.norm.sf(1.96 - (-10) / se_n) + stats.norm.cdf(-1.96 - (-10) / se_n)
    print(f"n = {n:3d}: power to detect a −10 mg/dL effect = {pw:.3f}")

🔍 **What just happened?** *Left:* each curve is one experimental design. Moving along a curve is moving the cut-off
(trading α for β); jumping to a lower curve — a bigger separation $\sqrt n\,\Delta/\sigma$ — makes **both** errors smaller
at once. *Right:* power curves of the two-sided $z$-test. At a true effect of 0 every curve passes through **0.05** (the
dotted line): when $H_0$ is true, "power" is just α. The curves rise towards 1 as the effect grows, and faster for larger
$n$. For a −10 mg/dL effect the power is **0.240 with 25 patients, 0.705 with 100 and 0.999 with 400**.

### How many patients? Deriving the sample-size formula

Suppose we want a two-sided test at level α to have power $1-\beta$ for an effect $\Delta>0$ (the case $\Delta<0$ is the
mirror image). The test rejects when $Z = (\bar X-\mu_0)/(\sigma/\sqrt n) \ge z_{1-\alpha/2}$ (we ignore the opposite tail,
whose probability is negligible when the power is high). If the truth is $\mu^* = \mu_0+\Delta$, then
$Z\sim\mathcal N\big(\sqrt n\,\Delta/\sigma,\ 1\big)$, so

$$\begin{aligned}
\text{power} = P\big(Z\ge z_{1-\alpha/2}\big) &= 1-\Phi\Big(z_{1-\alpha/2}-\sqrt n\,\tfrac{\Delta}{\sigma}\Big) \;\overset{!}{=}\; 1-\beta\\
\iff\quad z_{1-\alpha/2}-\sqrt n\,\tfrac{\Delta}{\sigma} &= \Phi^{-1}(\beta) = -z_{1-\beta}\\
\iff\quad \sqrt n &= \frac{(z_{1-\alpha/2}+z_{1-\beta})\,\sigma}{\Delta}\\
\iff\quad n &= \left(\frac{(z_{1-\alpha/2}+z_{1-\beta})\,\sigma}{\Delta}\right)^2 .
\end{aligned}$$

*In words:* the required sample size grows with the square of the biological spread $\sigma$ and shrinks with the square of
the effect $\Delta$ you want to detect: halving the effect quadruples the number of patients. Here $z_q$ is the $q$-quantile
of the standard normal ($z_{0.975} = 1.96$, $z_{0.8}=0.8416$), and we used $\Phi^{-1}(\beta) = -z_{1-\beta}$ (symmetry of
the bell curve).

In [ ]:
# ── 5.4 Plan the study: patients needed for 80 % power, then check by simulation ──
def sample_size_z(delta, sigma, alpha=0.05, power=0.8):
    """Smallest n for a two-sided one-sample z-test at level alpha to reach `power` for an effect delta."""
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    return int(np.ceil(((z_a + z_b) * sigma / abs(delta)) ** 2))


n_needed = sample_size_z(-10, SIGMA)
print(f"z(0.975) = {stats.norm.ppf(0.975):.4f}, z(0.80) = {stats.norm.ppf(0.80):.4f}")
print(f"patients needed for 80 % power against a −10 mg/dL effect: n = {n_needed}")
for d in [-20, -10, -5]:
    print(f"   effect {d:+d} mg/dL → n = {sample_size_z(d, SIGMA)}")

rng_ss = np.random.default_rng(SEED + 54)
sims = rng_ss.normal(MU0 - 10, SIGMA, (20_000, n_needed))
print(f"simulated power with n = {n_needed}: {reject_rule(sims).mean():.3f}")

🔍 **What just happened?** With $z_{0.975}=1.96$ and $z_{0.8}=0.8416$, the formula asks for
$\big((1.96+0.8416)\cdot40/10\big)^2 = 125.6$, i.e. **126 patients** — and a simulation of 20,000 studies of that size
confirms a power of about 0.80. Halving the effect we want to detect (−5 mg/dL) quadruples the requirement to 503
patients; doubling it (−20) needs only 32. Our 100-patient study was slightly under-powered for a 10 mg/dL effect.

### 📘 Lecture notes — the error picture and the trade-off

- **Picture:** two densities of the statistic $\omega$: $H_0$ centred at the known $\mu_0$, $H_1$ at the (usually unknown)
  true $\mu^*$. **Cut-offs** (green lines) split the axis into "retain $H_0$" and "choose $H_1$".
  Type I error = $H_0$-area in the reject region (blue); Type II = $H_1$-area in the retain region (red);
  power = $H_1$-area in the reject region (green hatching).
- **Two-sided** $H_0:\mu=x$ vs $H_1:\mu\ne x$: two cut-offs, two blue tails. **One-sided** $H_0:\mu\le x$ vs $H_1:\mu>x$:
  one cut-off, one blue tail.
- **Formulas** (statistic $\sim\mathcal N(\mu_0,s^2)$ or $\mathcal N(\mu^*,s^2)$): $\alpha$ = null area outside
  $[c_{lo},c_{hi}]$; $\beta = \Phi\big(\frac{c_{hi}-\mu^*}{s}\big)-\Phi\big(\frac{c_{lo}-\mu^*}{s}\big)$.
- **Trade-off (no free lunch):** moving the cut-off so that the Type I error goes down makes the Type II error go up and
  the power go down. As $\mu^*$ and $\mu_0$ move apart (or $n$ grows), β goes down at a fixed α and the signal becomes
  easier to detect; α itself is fixed by the cut-off.
- **Separation** in standard errors $=\sqrt n\,\Delta/\sigma$. **Sample size:** $n = \big((z_{1-\alpha/2}+z_{1-\beta})\sigma/\Delta\big)^2$;
  cholesterol example ($\sigma=40$, $\Delta=10$, α = 0.05, power 0.8): $n = 126$.

> 🎓 **Exam tip.** Be able to shade α, β and power on a sketch of two bell curves for both a two-sided and a one-sided test, and to say which areas change when you move the cut-off, $\mu^*$, or $n$.

### 🌉 Bridge — How many mice? Power analysis in experimental design

Animal-welfare rules (the "3Rs": replace, reduce, refine) and funding agencies ask biologists to *justify* their group
sizes with a power calculation, and journals increasingly ask for it in the methods. The same algebra works for comparing
two groups (knockout vs wild type, $n$ animals each): the difference of two means has standard error
$\sigma\sqrt{2/n}$, so the formula gains a factor 2,
$n_{\text{per group}} = 2\big((z_{1-\alpha/2}+z_{1-\beta})\,\sigma/\Delta\big)^2$. For an effect of one standard deviation
($\Delta/\sigma = 1$, a "large" effect in many phenotyping studies) at α = 0.05 and 80 % power that is
$2\times2.8016^2 = 15.7$, i.e. **16 mice per group** (17 with the slightly heavier-tailed $t$ distribution of
Section 7). Under-powered studies are a recognised problem: they miss real effects, and the "significant" results they
do produce tend to overestimate effect sizes (Button *et al.* 2013). In machine learning the same thinking appears when
deciding how many labelled samples a model needs ([BiologyToML Module 23 · Model selection](../../../../ml/tutorial/BiologyToML/notebooks/23_model_selection.ipynb)).

✏️ **Your turn.** (a) Use `sample_size_z` to find how many patients detect a −10 mg/dL effect with 90 % power. (b) By
hand: if $\sigma$ were 20 mg/dL instead of 40 (a more homogeneous patient group), how would $n$ change — and why is
"measure a more homogeneous group" sometimes cheaper than "measure more patients"?

---
## 6 · Performing a test in three steps: the $z$-test

> 📍 **Where we are.** Sections 3–5 fixed a rule and studied its errors. Now we turn the rule into the everyday recipe
> that ends with a $p$-value, and apply it to our 100 patients.

Let us assume we have data $\mathcal D = X_1, X_2,\dots,X_n$ that all follow the same distribution with known form
$p(X_i\mid\theta)$. We perform a test in **3 steps**:

1. **Compute a test statistic** — a function of the data that is appropriate for the distribution:
   $T = r(X_1, X_2,\dots,X_n)$.
2. **Compute a $p$-value.**
3. **For a desired significance level $\alpha$** and the $p$-value, **decide** whether to retain or reject $H_0$.

**The cholesterol example.** We test the efficacy of the drug, assuming we know the typical variation of cholesterol among
humans ($\sigma = 40$ mg/dL) and that it does not change between the conditions.

**Step 1 — the statistic.** We assume the samples come from a Gaussian, $X_1,\dots,X_n\sim\mathcal N(\mu_0,\sigma^2)$ under
$H_0$, so the natural statistic is the sample mean $\overline{X_n}$. We run a **two-sided** test (we do not know whether the
drug will lower or raise cholesterol; we want to know if it changes): $H_0:\mu=\mu_0$ vs $H_1:\mu\ne\mu_0$.

Under the null hypothesis, by the properties of Modules 21–22 ($E[\overline{X_n}]=\mu_0$, $\operatorname{Var}\overline{X_n}
= \sigma^2/n$, and a sum of independent normals is normal):

$$\overline{X_n}\sim\mathcal N\!\Big(\mu_0,\ \frac{\sigma^2}{n}\Big)\qquad\Longrightarrow\qquad
Z \;=\; \frac{\overline{X_n}-\mu_0}{\sigma/\sqrt n}\;\sim\;\mathcal N(0,1).$$

*In words:* if the drug does nothing, the sample mean scatters around 210 with standard error $\sigma/\sqrt n$; measuring
its distance from 210 in standard errors gives a standard normal variable — the same yardstick for every study.

**Step 2 — the $p$-value.** The $p$-value is the probability, computed **under the null hypothesis**, of seeing a test
statistic **at least as extreme** as the one we observed. For the two-sided test, "extreme" means far from 0 in either
direction:

$$p \;=\; P\Big(|Z| \ge |z_{\text{obs}}|\ \Big|\ H_0\Big), \qquad z_{\text{obs}} = \frac{\bar x_n-\mu_0}{\sigma/\sqrt n},
\qquad\text{so}\qquad p = 2\big(1-\Phi(|z_{\text{obs}}|)\big).$$

*In words:* imagine thousands of studies of untreated people; $p$ is the fraction of them whose sample mean would be at
least as far from 210 as ours. Small $p$ = our result would be surprising if $H_0$ were true.

⚠️ **The absolute value goes on both sides.** Without $|\cdot|$ around the *observed* statistic, a negative $z_{\text{obs}}$
(our drug *lowers* cholesterol) would give $P(|Z| > -2.2) = 1$ — nonsense. Always compare $|Z|$ with $|z_{\text{obs}}|$.

🧮 **Math refresher — absolute-value inequalities.** $|Z|\ge c$ (with $c\ge0$) means $Z\ge c$ **or** $Z\le -c$. Because the
standard normal is symmetric, $P(|Z|\ge c) = P(Z\ge c)+P(Z\le-c) = 2\,P(Z\ge c) = 2(1-\Phi(c))$.

In [ ]:
# ── 6.1 The z-test by hand, checked two ways ──
def z_test_mean(x, mu0, sigma, alternative="two-sided"):
    """One-sample z-test for a mean with KNOWN sigma. Returns (z, p).
    alternative: 'two-sided' (H1: mu ≠ mu0), 'less' (H1: mu < mu0) or 'greater' (H1: mu > mu0)."""
    x = np.asarray(x, dtype=float)
    z = (x.mean() - mu0) / (sigma / np.sqrt(x.size))          # Step 1: the test statistic
    if alternative == "two-sided":                              # Step 2: the p-value under H0: Z ~ N(0, 1)
        p = 2 * stats.norm.sf(abs(z))
    elif alternative == "less":
        p = stats.norm.cdf(z)
    else:
        p = stats.norm.sf(z)
    return z, p


z_obs, p_obs = z_test_mean(cholesterol, MU0, SIGMA)
print(f"Step 1: x̄ = {cholesterol.mean():.2f}, standard error = {SIGMA / np.sqrt(N_PATIENTS):.1f}, "
      f"z = ({cholesterol.mean():.2f} − {MU0:.0f}) / {SIGMA / np.sqrt(N_PATIENTS):.1f} = {z_obs:.4f}")
print(f"Step 2: p = 2·(1 − Φ(|z|)) = 2·(1 − {stats.norm.cdf(abs(z_obs)):.5f}) = {p_obs:.4f}")

# check 1: the same number from the CDF directly
print(f"check with Φ:  P(Z ≤ −|z|) + P(Z ≥ |z|) = {stats.norm.cdf(-abs(z_obs)) + stats.norm.sf(abs(z_obs)):.4f}")
# check 2: brute force — 200,000 studies of untreated patients
rng_null = np.random.default_rng(SEED + 61)
z_null = (rng_null.normal(MU0, SIGMA / np.sqrt(N_PATIENTS), 200_000) - MU0) / (SIGMA / np.sqrt(N_PATIENTS))
print(f"check by simulation: fraction of null studies with |Z| ≥ {abs(z_obs):.3f} = {np.mean(np.abs(z_null) >= abs(z_obs)):.4f}")
print(f"Step 3: at α = 0.05 → {'reject H0' if p_obs <= 0.05 else 'retain H0'};  "
      f"at α = 0.01 → {'reject H0' if p_obs <= 0.01 else 'retain H0'}")

In [ ]:
# ── 6.2 Picture of the p-value: the two tails of N(0, 1) beyond ±|z_obs| ──
zz = np.linspace(-4, 4, 801)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(zz, stats.norm.pdf(zz), color="tab:blue", label="null distribution of Z: N(0, 1)")
shade(ax, zz, stats.norm.pdf(zz), np.abs(zz) >= abs(z_obs), color="tab:purple", alpha=0.5,
      label=f"p-value = {p_obs:.4f} (both tails)")
ax.axvline(z_obs, ymax=0.75, color="k", ls="--", lw=1.2, label=f"observed z = {z_obs:.2f}")
for zc in (-1.96, 1.96):
    ax.axvline(zc, color="tab:green", lw=2, ymax=0.5)
ax.text(1.96, 0.21, " ±1.96: cut-offs\n for α = 0.05", color="tab:green", fontsize=7)
ax.set_ylim(0, 0.55)
ax.set_xlabel("z"); ax.set_ylabel("density"); ax.legend(loc="upper left", fontsize=7)
ax.set_title("two-sided p-value of the cholesterol study")
plt.tight_layout(); plt.show()

🔍 **What just happened?** **Step 1:** the sample mean 201.07 mg/dL is $-8.93$ mg/dL from 210, i.e. **z = −2.2325**
standard errors. **Step 2:** the probability that untreated patients would give a sample mean at least this far from
210 (in either direction) is **p = 0.0256** — the purple area in both tails. The simulation of 200,000 null studies
gives the same fraction (0.0255, up to Monte-Carlo noise). **Step 3:** at α = 0.05 we **reject $H_0$** (the drug changed
cholesterol); at the stricter α = 0.01 we would retain it. Notice that the observed $z$ lies beyond the green cut-off
−1.96: "$p\le\alpha$" and "$|z|\ge z_{1-\alpha/2}$" are the same decision, read on two different scales.

**Step 3 — deciding.** Given the $p$-value and the significance level α, decide:

* **choose $H_0$ ("retain $H_0$")** if $p > \alpha$;
* **choose $H_1$ ("reject $H_0$")** if $p \le \alpha$. Cut-offs α = 0.01, 0.05 and 0.1 are frequently used.

Depending on how strict we want to be, a common rule of thumb for **reading a $p$-value** as evidence is:

| $p$-value | interpretation |
|---|---|
| < 0.01 | very strong evidence against $H_0$ |
| 0.01 – 0.05 | strong evidence against $H_0$ |
| 0.05 – 0.1 | weak evidence against $H_0$ |
| > 0.1 | little to no evidence against $H_0$ |

Our $p = 0.026$ is "strong evidence against $H_0$". Why does "$p\le\alpha$" give a test with Type I error rate exactly α?
Because **under $H_0$ the $p$-value is uniformly distributed on $[0,1]$** for a continuous statistic — so
$P(p\le\alpha\mid H_0) = \alpha$. Let us see it.

In [ ]:
# ── 6.3 p-values of 20,000 studies: uniform under H0, piled up near 0 under H1 ──
z_h0 = (world_h0.mean(axis=1) - MU0) / (SIGMA / np.sqrt(N_PATIENTS))
z_h1 = (world_h1.mean(axis=1) - MU0) / (SIGMA / np.sqrt(N_PATIENTS))
p_h0, p_h1 = 2 * stats.norm.sf(np.abs(z_h0)), 2 * stats.norm.sf(np.abs(z_h1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
axes[0].hist(p_h0, bins=20, range=(0, 1), color="tab:blue", edgecolor="white")
axes[0].set_title("H0 true (μ = 210): p-values are uniform")
axes[1].hist(p_h1, bins=20, range=(0, 1), color="tab:red", edgecolor="white")
axes[1].set_title("H1 true (μ = 200): p-values pile up near 0")
for ax in axes:
    ax.axvline(0.05, color="k", ls=":", lw=1)
    ax.set_xlabel("p-value")
axes[0].set_ylabel("number of simulated studies")
plt.tight_layout(); plt.show()
for a in (0.01, 0.05, 0.10):
    print(f"P(p ≤ {a:.2f} | H0) ≈ {np.mean(p_h0 <= a):.4f}     P(p ≤ {a:.2f} | H1) = power ≈ {np.mean(p_h1 <= a):.4f}")

🔍 **What just happened?** Under $H_0$ the histogram is flat: each of the 20 bins holds about 1,000 of the 20,000
studies, and the fraction with $p\le0.05$ is ≈ 0.05 (likewise ≈ 0.01 and ≈ 0.10 for the other levels). That is the
Type I error guarantee. Under $H_1$ the $p$-values crowd towards 0: the fraction below 0.05 is the power, ≈ 0.70 again.
A $p$-value is therefore itself a random variable — repeat the study and you get a different one.

⚠️ **What a $p$-value is NOT.** It is not the probability that $H_0$ is true; it is not the probability that the result
"is due to chance"; it is not the size or importance of the effect; and $p>0.05$ does not show that $H_0$ is true. It is
only $P(\text{data at least this extreme}\mid H_0)$ — a probability *given* $H_0$, just as the sensitivity of a
diagnostic test is a probability *given* the disease (Module 17).

**One-sided tests.** For a one-sided test such as $H_0:\mu\ge\mu_0$ vs $H_1:\mu<\mu_0$ ("the drug *lowers*
cholesterol"), the only thing that changes is Step 2: only the tail in the direction of $H_1$ counts,

$$p = P\big(Z\le z_{\text{obs}}\mid H_0\big) = \Phi(z_{\text{obs}}), \qquad\text{or, for } H_1:\mu>\mu_0,\qquad p = P\big(Z\ge z_{\text{obs}}\mid H_0\big) = 1-\Phi(z_{\text{obs}}).$$

*In words:* a result counts as extreme only if it is extreme *in the predicted direction*. One-sided tests are usually
**more powerful in that direction** because the whole α is spent in one tail (cut-off 1.645 instead of 1.96) — but they
have essentially no power in the other direction.

In [ ]:
# ── 6.4 One-sided vs two-sided: p-values and power in both directions ──
z1, p_less = z_test_mean(cholesterol, MU0, SIGMA, alternative="less")
_, p_greater = z_test_mean(cholesterol, MU0, SIGMA, alternative="greater")
print(f"H1: μ ≠ 210 → p = {p_obs:.4f}    H1: μ < 210 → p = {p_less:.4f}    H1: μ > 210 → p = {p_greater:.4f}")

se = SIGMA / np.sqrt(N_PATIENTS)
eff = np.linspace(-25, 25, 401)
pw_two = stats.norm.sf(1.96 - eff / se) + stats.norm.cdf(-1.96 - eff / se)
pw_less = stats.norm.cdf(-stats.norm.isf(0.05) - eff / se)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(eff, pw_two, label="two-sided (H1: μ ≠ 210)")
ax.plot(eff, pw_less, label="one-sided (H1: μ < 210)")
ax.axhline(0.05, color="0.5", ls=":", lw=1)
ax.set_xlabel("true effect μ* − 210 (mg/dL)"); ax.set_ylabel("power"); ax.legend()
ax.set_title("power of one- and two-sided z-tests (n = 100, α = 0.05)")
plt.tight_layout(); plt.show()
i10 = np.argmin(np.abs(eff + 10))
print(f"power at −10 mg/dL: two-sided {pw_two[i10]:.3f}, one-sided {pw_less[i10]:.3f};  "
      f"at +10 mg/dL: two-sided {pw_two[np.argmin(np.abs(eff - 10))]:.3f}, one-sided {pw_less[np.argmin(np.abs(eff - 10))]:.5f}")

🔍 **What just happened?** For our data ($z=-2.23$) the one-sided $p$ for "the drug lowers cholesterol" is **0.0128**,
exactly half of the two-sided 0.0256; the opposite one-sided alternative ("the drug raises cholesterol") gets
$p = 0.987$ — no evidence at all. The power curves show the price: for a true −10 mg/dL effect the one-sided test has
power **0.804** versus 0.705 for the two-sided test, but if the drug unexpectedly *raised* cholesterol by 10 mg/dL the
one-sided test would almost never notice (power 0.00002), while the two-sided test would (0.705).

**Main takeaways of the recipe.** When performing tests, the main things we need to determine are:

* the **distribution we think the data came from** (here normal with known σ);
* the **test statistic** that is appropriate for those samples (here $\overline{X_n}$, standardised to $Z$);
* the **distribution of the test statistic under $H_0$** — it may be different from the one of the data (the data are
  $\mathcal N(\mu,\sigma^2)$; $Z$ is $\mathcal N(0,1)$; in Section 7 it will be a $t$ distribution).

### 📘 Lecture notes — performing a test: the $z$-test

- **Three steps.** Data $\mathcal D = X_1,\dots,X_n$ i.i.d. with known $p(X_i\mid\theta)$. (1) test statistic
  $T = r(X_1,\dots,X_n)$; (2) $p$-value; (3) for a significance level α, retain or reject $H_0$.
- **$z$-test** (normal data, known σ). Under $H_0:\mu=\mu_0$: $\overline{X_n}\sim\mathcal N(\mu_0,\sigma^2/n)$, so
  $Z = \frac{\overline{X_n}-\mu_0}{\sigma/\sqrt n}\sim\mathcal N(0,1)$.
- **$p$-value** = probability under $H_0$ of a statistic at least as extreme as observed. Two-sided:
  $p = P\big(|Z|\ge\big|\frac{\bar x_n-\mu_0}{\sigma/\sqrt n}\big|\big) = 2(1-\Phi(|z|))$ (absolute value on **both** sides).
  One-sided $H_1:\mu>\mu_0$: $p = P\big(Z\ge\frac{\bar x_n-\mu_0}{\sigma/\sqrt n}\big)$; for $H_1:\mu<\mu_0$ use the lower tail.
- **Decision:** retain $H_0$ if $p>\alpha$, reject if $p\le\alpha$ (α = 0.01, 0.05, 0.1 common); equivalently reject if
  $|z|\ge z_{1-\alpha/2}$ (1.96 for α = 0.05). **Evidence table:** < 0.01 very strong; 0.01–0.05 strong; 0.05–0.1 weak;
  > 0.1 little or none (evidence *against* $H_0$).
- **Worked example:** $n=100$, $\sigma=40$, $\bar x=201.07$: $z=-2.23$, two-sided $p=0.026$ → reject at 0.05, not at 0.01;
  one-sided ($\mu<210$) $p = 0.013$.
- **Facts:** under $H_0$, $p\sim$ Uniform(0,1), so $P(p\le\alpha\mid H_0)=\alpha$. One-sided tests are more powerful in the
  predicted direction (they integrate over one region only) and blind in the other.
- **Takeaways:** decide the distribution of the data, the statistic, and the statistic's distribution under $H_0$.

> 🎓 **Exam tip.** Write the $p$-value as a probability **under $H_0$** of a statistic **at least as extreme** as the observed one. "$p$ = probability that $H_0$ is true" earns no credit.

### 🌉 Bridge — What fraction of "significant" findings are false? (Bayes' rule meets testing)

α controls $P(\text{reject}\mid H_0)$. A biologist reading a paper wants the reverse, $P(H_0\mid\text{reject})$ — and
Bayes' rule (Module 17) says that depends on how many of the tested hypotheses were true to begin with. Suppose only
10 % of candidate drugs entering a screen really work, each is tested at α = 0.05 with power 0.7. Then

$$P(H_0\mid\text{reject}) = \frac{P(\text{reject}\mid H_0)P(H_0)}{P(\text{reject}\mid H_0)P(H_0)+P(\text{reject}\mid H_1)P(H_1)}
= \frac{0.05\cdot0.9}{0.05\cdot0.9+0.7\cdot0.1} = \frac{0.045}{0.115}\approx 0.39 .$$

*In words:* in this screen almost 4 out of 10 "significant" drugs would be duds, even though every single test had a 5 %
false-positive rate. The fraction of false discoveries among all discoveries — the **false discovery rate** — is the
central idea of [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb), where we test 400 genes at once. Its practical consequences for reproducibility are discussed in
[BiologyToML Module 42 · Trustworthy ML](../../../../ml/tutorial/BiologyToML/notebooks/42_trustworthy_ml.ipynb).

✏️ **Your turn.** Recompute $P(H_0\mid\text{reject})$ when 50 % of the candidates work, and when power is only 0.2 (a
small study) with 10 % working candidates. Which change hurts more?

### 🪙 Worked example — "Is it fair?" as a formal test

In [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) we counted **52 heads in 100 flips** and promised a formal test. Here it is, in the three steps. Code each
flip as $X_i = 1$ (heads) or $0$ (tails) and let $p$ be the probability of heads: $H_0: p = \tfrac12$ (a fair coin) versus
$H_1: p \ne \tfrac12$. If $H_0$ is true, one flip has mean $\tfrac12$ and standard deviation
$\sqrt{\tfrac12\cdot\tfrac12} = 0.5$ — *known*, because $H_0$ fixes the whole distribution, so no σ has to be estimated
and the $z$-test applies. By the central limit theorem ([Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb)), $\overline{X_{100}}$ is approximately
$\mathcal N\big(0.5,\ 0.5^2/100\big)$, with standard error $0.5/\sqrt{100} = 0.05$.

In [ ]:
# ── 6.5 Worked example: is the coin of Module 22 fair?  (52 heads in 100 flips) ──
n_flips, heads_obs, p_null = 100, 52, 0.5
sd_flip = np.sqrt(p_null * (1 - p_null))                     # SD of one flip if H0 is true (0.5; fixed by H0, not estimated)
se_coin = sd_flip / np.sqrt(n_flips)                         # standard error of the fraction of heads
z_coin = (heads_obs / n_flips - p_null) / se_coin            # Step 1: the test statistic
p_coin = 2 * stats.norm.sf(abs(z_coin))                      # Step 2: two-sided p-value (normal approximation)
print(f"Step 1: x̄ = {heads_obs}/{n_flips} = {heads_obs / n_flips:.2f}, standard error = {se_coin:.2f}, "
      f"z = ({heads_obs / n_flips:.2f} − {p_null}) / {se_coin:.2f} = {z_coin:.2f}")
print(f"Step 2: p = 2·(1 − Φ(|z|)) = {p_coin:.4f}")
print(f"Step 3: at α = 0.05 → {'reject H0' if p_coin <= 0.05 else 'retain H0'}")

# two cross-checks: the exact binomial test, and the normal approximation with a continuity correction
p_exact = stats.binomtest(heads_obs, n_flips, p_null).pvalue
z_cc = (abs(heads_obs - n_flips * p_null) - 0.5) / np.sqrt(n_flips * p_null * (1 - p_null))
print(f"exact binomial test: p = {p_exact:.4f};   normal approximation with continuity correction: p = {2 * stats.norm.sf(z_cc):.4f}")

🔍 **What just happened?** The fraction of heads, 0.52, is only $z = 0.40$ standard errors from 0.5, and the two-sided
$p$-value is **0.689**: in about 69 % of experiments with a perfectly fair coin the fraction of heads lands at least this
far from ½ — the same "about 69 %" as in [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb). Since $p > 0.05$ we **retain $H_0$** ("no evidence of unfairness" — not
"proof of fairness": 100 flips could not reliably tell a fair coin from one with heads probability 0.55). The exact
binomial test gives 0.764, a little larger than 0.689, because the number of heads is a whole number: 52 heads really
covers the interval from 51.5 to 52.5, and the *continuity correction* ($|52-50|-0.5 = 1.5$ instead of 2, so
$z = 0.3$) brings the normal approximation to 0.764 as well. All versions lead to the same decision. The same recipe
tests a 1 : 1 sex ratio or a Mendelian test cross (Aa × aa); for categories with more than two outcomes the $\chi^2$ test of
[Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb) takes over.

---
## 7 · When $\sigma$ is unknown: the $t$ distribution

> 📍 **Where we are.** The $z$-test needed the population standard deviation σ. In real experiments σ is almost never
> known — we estimate it from the same data. That small change alters the null distribution of the statistic, and the
> fix is one of the most used results in all of biology: Student's $t$ distribution.

Let us again assume $X_1,\dots,X_n\sim\mathcal N(\mu_0,\sigma^2)$ but this time we do **not** know σ. We define two
statistics — the sample mean and the sample variance (Module 21; with $n-1$, Bessel's correction):

$$\overline{X_n} = \frac1n\sum_{i=1}^n X_i, \qquad\qquad S^2 = \frac{1}{n-1}\sum_{i=1}^n\big(X_i-\overline{X_n}\big)^2 .$$

*In words:* $\overline{X_n}$ is the average; $S^2$ is the average squared deviation from it, dividing by $n-1$ so that it is
unbiased for $\sigma^2$. (Some texts write $\bar\sigma^2$ or $s^2$ for $S^2$.)

Replacing σ by $S$ in the $z$ statistic gives the ***t* statistic**, and (for normal data) its distribution under $H_0$ is
known exactly:

$$T = \frac{\overline{X_n}-\mu_0}{S/\sqrt n}\ \sim\ t_{n-1}.$$

*In words:* standardising with an *estimated* standard deviation gives a statistic that follows a $t$ distribution with
$n-1$ degrees of freedom — one less than the number of observations. W. S. Gosset derived it in 1908 while working on
small samples at the Guinness brewery in Dublin, publishing under the pen name "Student".

Why not just keep using $\mathcal N(0,1)$? Because $S$ is itself random: in a small sample it is sometimes much smaller
than σ, and then $T$ is inflated. Let us measure the damage with $n = 4$ (four mice per group is common!).

In [ ]:
# ── 7.1 What goes wrong if we plug S into the z-test with n = 4 (SIMULATED null data) ──
rng_t = np.random.default_rng(SEED + 71)
n_small = 4
null_small = rng_t.normal(0.0, 1.0, (100_000, n_small))              # H0 true: μ = 0
t_stat = null_small.mean(axis=1) / (null_small.std(axis=1, ddof=1) / np.sqrt(n_small))
print(f"fraction of null studies with |T| ≥ 1.96: {np.mean(np.abs(t_stat) >= 1.96):.4f}  (the z-test promised 0.05)")
print(f"theory: P(|t_3| ≥ 1.96) = {2 * stats.t.sf(1.96, df=n_small - 1):.4f}")
t_crit3 = stats.t.isf(0.025, df=n_small - 1)
print(f"correct two-sided cut-off for n = 4: t_(3, 0.975) = {t_crit3:.3f};  "
      f"fraction beyond it = {np.mean(np.abs(t_stat) >= t_crit3):.4f}")

grid = np.linspace(-6, 6, 601)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
bin_w = 0.1                                                          # density = count / (number of studies × bin width)
ax.hist(t_stat, bins=120, range=(-6, 6), weights=np.full(t_stat.size, 1 / (t_stat.size * bin_w)), color="0.75",
        label="T with n = 4 (100,000 null studies)")
ax.plot(grid, stats.norm.pdf(grid), color="tab:blue", label="N(0, 1)  (what the z-test assumes)")
ax.plot(grid, stats.t.pdf(grid, df=3), color="tab:red", ls="--", label="t with 3 degrees of freedom")
ax.set_xlabel("value of (x̄ − μ0) / (S/√n)"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("estimating σ fattens the tails")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With $n=4$ and $S$ in place of σ, about **14.5 %** of null studies give $|T|\ge1.96$ — almost
three times the promised 5 %. The histogram has the same centre as $\mathcal N(0,1)$ but much heavier tails, and the
$t_3$ curve fits it perfectly. The correct cut-off is $t_{3,\,0.975} = 3.182$, beyond which exactly 5 % of null studies
fall. Using the normal distribution with an estimated σ in small samples is a recipe for false positives.

### The $t$ distribution

The $t$ distribution has one parameter: $\nu$, the **degrees of freedom**. If $X\sim t_\nu$, its density is

$$p(X=x) \;=\; \frac{\Gamma\!\left(\frac{\nu+1}{2}\right)}{\sqrt{\pi\nu}\;\Gamma\!\left(\frac{\nu}{2}\right)}\left(1+\frac{x^2}{\nu}\right)^{-\frac{\nu+1}{2}} .$$

*In words:* the shape is the bell-like curve $\big(1+x^2/\nu\big)^{-(\nu+1)/2}$ — which, unlike the Gaussian
$e^{-x^2/2}$, decays only like a power of $x$, so extreme values are more likely; the constant in front makes the total
area 1. (We write $p(X=x)$ for the density, as in Module 18 — remember it is a density, not a probability.)

🧮 **Math refresher — the Gamma function.** $\Gamma(z) = \int_0^\infty t^{z-1}e^{-t}\,dt$ extends the factorial to
non-integers: $\Gamma(n) = (n-1)!$ for positive integers ($\Gamma(1)=1$, $\Gamma(2)=1$, $\Gamma(3)=2$, $\Gamma(4)=6$),
$\Gamma(z+1) = z\,\Gamma(z)$, and $\Gamma(\tfrac12) = \sqrt\pi$. In Python: `math.gamma`. You met it in the Beta
distribution of Module 24.

In [ ]:
# ── 7.2 The t density by hand, checked with SciPy and by integration ──
from math import exp, gamma, lgamma, pi, sqrt


def t_density(x, nu):
    """Density of Student's t distribution with nu degrees of freedom (the formula above).
    The ratio of Gamma functions is computed as exp(log Γ − log Γ): Γ(500) alone has more than
    1,000 digits and would overflow a float, but the ratio is a modest number."""
    const = exp(lgamma((nu + 1) / 2) - lgamma(nu / 2)) / sqrt(pi * nu)
    return const * (1 + np.asarray(x, dtype=float) ** 2 / nu) ** (-(nu + 1) / 2)


print("Γ(1/2) =", round(gamma(0.5), 6), " √π =", round(sqrt(pi), 6), "  Γ(4) = 3! =", gamma(4))
for nu in [1, 5, 30]:
    mine = t_density([0.0, 1.0, 3.0], nu)
    print(f"ν = {nu:2d}: by hand {np.round(mine, 5)}   scipy {np.round(stats.t.pdf([0, 1, 3], nu), 5)}   "
          f"∫ density = {integrate.quad(t_density, -np.inf, np.inf, args=(nu,))[0]:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
xs = np.linspace(-5, 5, 501)
for ax, logy in zip(axes, [False, True]):
    ax.plot(xs, stats.norm.pdf(xs), color="tab:blue", label="N(0, 1)")
    ax.plot(xs, t_density(xs, 1), color="tab:red", label="t₁")
    ax.plot(xs, t_density(xs, 5), color="tab:orange", label="t₅")
    ax.plot(xs, t_density(xs, 30), color="tab:green", ls="--", label="t₃₀")
    ax.set_xlabel("x"); ax.set_ylabel("density"); ax.legend()
    if logy:
        ax.set_yscale("log"); ax.set_ylim(1e-6, 1)
        ax.set_title("log scale: the tails")
    else:
        ax.set_title("t distributions vs the standard normal")
plt.tight_layout(); plt.show()
for nu in [1, 5, 30]:
    print(f"P(|T| > 3) for t_{nu}: {2 * stats.t.sf(3, nu):.4f}")
print(f"P(|Z| > 3) for N(0,1): {2 * stats.norm.sf(3):.4f}")

🔍 **What just happened?** Our hand-written density agrees with `scipy.stats.t.pdf` to every printed digit and integrates
to 1 for each ν. At the centre, $t_1$ is lowest (0.318 = $1/\pi$) and the normal highest (0.399); the curves cross, and
in the tails the order reverses. The log-scale panel shows the tails: $P(|T|>3)$ is **0.2048** for $t_1$, **0.0301**
for $t_5$, 0.0054 for $t_{30}$ and only **0.0027** for $\mathcal N(0,1)$. The $t$ distribution has the shape of the normal
but a thicker tail — and $t_{30}$ is already hard to tell from the normal on the linear scale.

**As $\nu\to\infty$ the $t$ distribution becomes the standard normal.** Intuition: with many observations $S$ is an
excellent estimate of σ, so standardising by $S$ or by σ hardly differs. (Mathematically,
$\big(1+x^2/\nu\big)^{-(\nu+1)/2}\to e^{-x^2/2}$, the same limit as $(1+a/\nu)^\nu\to e^{a}$ from Module 14.)

In [ ]:
# ── 7.3 ν → ∞: the largest gap between the t density and the normal density ──
xs_fine = np.linspace(-6, 6, 2001)
for nu in [1, 2, 5, 10, 30, 100, 1000]:
    gap = np.max(np.abs(t_density(xs_fine, nu) - stats.norm.pdf(xs_fine)))
    print(f"ν = {nu:5d}: max |t_ν density − N(0,1) density| = {gap:.5f}   two-sided 5% cut-off = {stats.t.isf(0.025, nu):.3f}")
print(f"normal:                                               two-sided 5% cut-off = {stats.norm.isf(0.025):.3f}")

# why n − 1 degrees of freedom: the n deviations from the mean always sum to zero
sample5 = np.array([4.0, 7.0, 6.0, 9.0, 4.0])
dev = sample5 - sample5.mean()
print(f"\ndeviations from the mean {dev} sum to {dev.sum():.1f} → only n − 1 = 4 of them are free")

🔍 **What just happened?** The biggest difference between the $t_\nu$ and the normal densities falls from 0.099 at ν = 1
to 0.0001 at ν = 1000, and the two-sided 5 % cut-off falls from 12.706 (ν = 1) to 2.042 (ν = 30) and 1.963 (ν = 1000),
approaching the normal's 1.960. In practice, for ν above about 30 the two differ only in the second decimal.

**Why "degrees of freedom" = $n-1$?** The deviations $X_i-\overline{X_n}$ always sum to zero (here $-2, 1, 0, 3, -2$).
Once you know four of them, the fifth is determined — so $S^2$ is built from only $n-1$ "free" pieces of information. We
assume there is one less degree of freedom than there are observations because one was spent on estimating the mean.

### Reading a $t$ table

The probabilities of the $t$ distribution have no simple closed form, so before computers everyone used **lookup tables**.
They are still printed in textbooks and exams. We generate one with `stats.t.isf` (the inverse of the upper-tail
probability): each entry is the value $t^*$ with $P(T\ge t^*) = $ the column heading, for $T\sim t_\nu$ — so this is a
table of **one-tailed** upper probabilities.

In [ ]:
# ── 7.4 Generate a t table: critical values t* with P(T ≥ t*) = column heading ──
tail_probs = [0.25, 0.20, 0.15, 0.10, 0.05, 0.025, 0.01, 0.005, 0.001]
dofs = list(range(1, 31)) + [40, 60, 120]
t_table = pd.DataFrame([[stats.t.isf(a, nu) for a in tail_probs] for nu in dofs],
                       index=pd.Index(dofs, name="ν (df)"), columns=[str(a) for a in tail_probs])
t_table.loc["∞ (normal)"] = [stats.norm.isf(a) for a in tail_probs]
pd.set_option("display.width", 120)
print("one-tailed upper probabilities P(T ≥ t*) → critical values t*")
print(t_table.round(3).to_string())

🔍 **What just happened?** Row ν = 1 starts 1.000, 1.376, 1.963, 3.078, 6.314, 12.706 …; row ν = 10 reads 0.700, 0.879,
1.093, 1.372, 1.812, 2.228, 2.764, 3.169, 4.144; the last row is the standard normal (0.674 … 1.645, 1.960, 2.326, 2.576,
3.090). Each row decreases towards the normal row as ν grows — the heavier tails of small samples need larger cut-offs.

**How to use the table.** Find your degrees of freedom in the left column; in that row find the column with the next
**smaller** number than your statistic $|t|$; read the probability at the top of that column. Since your $t$ is a little
bigger than that table value, your one-tailed $p$ is a little *smaller* than the heading (e.g. $p<0.01$). If your $t$ is
to the right of all numbers in the row, $p$ is less than the right-most probability. **For a two-sided test double the
probability** (the table is one-tailed).

*Worked example:* $t = 2.9$ with ν = 11. Row 11 contains 2.718 (column 0.01) and 3.106 (column 0.005); 2.9 lies between
them, so the one-tailed $p$ is between 0.005 and 0.01, and the two-sided $p$ between 0.01 and 0.02.

In [ ]:
# ── 7.5 Check the table reading with the exact tail probability ──
t_ex, nu_ex = 2.9, 11
row = t_table.loc[nu_ex]
smaller = row[row <= t_ex]
print(f"row ν = {nu_ex}: largest table value below t = {t_ex} is {smaller.iloc[-1]:.3f} in column {smaller.index[-1]}")
print(f"exact one-tailed p = P(T ≥ {t_ex}) = {stats.t.sf(t_ex, nu_ex):.4f};  two-sided p = {2 * stats.t.sf(t_ex, nu_ex):.4f}")
print(f"(a normal table would have said two-sided p = {2 * stats.norm.sf(t_ex):.4f} — too small)")

🔍 **What just happened?** The table says "one-tailed $p$ just below 0.01"; the exact value is **0.0072** (two-sided
**0.0144**), inside the bracket we read off. Pretending the statistic were normal would have given a two-sided $p$ of
0.0037 — four times too small. With software we compute exact $p$-values with `stats.t.sf`; the table remains useful for
quick checks and for exams.

### 📘 Lecture notes — the $t$ distribution and the one-sample $t$-test

- **When σ is unknown**, estimate it: $\overline{X_n} = \frac1n\sum X_i$, $S^2 = \frac{1}{n-1}\sum(X_i-\overline{X_n})^2$.
  For normal data, $T = \frac{\overline{X_n}-\mu_0}{S/\sqrt n}\sim t_{n-1}$ under $H_0:\mu=\mu_0$.
- **The $t$ distribution** has one parameter, the degrees of freedom ν:
  $p(x) = \frac{\Gamma(\frac{\nu+1}{2})}{\sqrt{\pi\nu}\,\Gamma(\frac\nu2)}\big(1+\frac{x^2}{\nu}\big)^{-\frac{\nu+1}{2}}$.
  Symmetric around 0, bell-shaped, **thicker tails** than the normal; $t_\nu\to\mathcal N(0,1)$ as $\nu\to\infty$.
- **Degrees of freedom** $n-1$: one less than the number of observations (the deviations from the mean sum to zero).
- **Using a normal cut-off with estimated σ** inflates false positives in small samples ($n=4$: 14.5 % instead of 5 %).
- **Tables** list critical values $t^*$ with $P(T\ge t^*)$ = column heading (one-tailed). Find the row of your ν, the
  column with the next smaller value than your $|t|$, read the probability; double it for a two-sided test; if $|t|$ is
  beyond the last column, $p$ is smaller than the last heading. Example: $t=2.9$, ν = 11 → $0.005 < p_{\text{one}} < 0.01$.

> 🎓 **Exam tip.** Remember the three changes from $z$ to $t$: $S$ (with $n-1$) replaces σ, $t_{n-1}$ replaces $\mathcal N(0,1)$, and the cut-offs get larger (2.228 instead of 1.96 for ν = 10, two-sided α = 0.05).

✏️ **Your turn.** Using only the table: (a) a one-sample test with $n=16$ gives $t=2.4$; bracket the two-sided $p$.
(b) With ν = 5, what $|t|$ is needed for two-sided significance at α = 0.01? Check both with `stats.t`.

---
## 8 · One-sample and paired $t$-tests: the cholesterol trial

> 📍 **Where we are.** We have the $t$ distribution. First we redo the 100-patient study without pretending to know σ;
> then we meet the most important design trick of clinical biology — measuring the *same* individuals twice.

In [ ]:
# ── 8.1 The 100 patients again, now with σ estimated from the data (one-sample t-test) ──
def one_sample_t(x, mu0):
    """One-sample t-test of H0: mu = mu0 (two-sided). Returns (t, df, p)."""
    x = np.asarray(x, dtype=float)
    n = x.size
    t = (x.mean() - mu0) / (x.std(ddof=1) / np.sqrt(n))
    return t, n - 1, 2 * stats.t.sf(abs(t), n - 1)


t1, df1, p1 = one_sample_t(cholesterol, MU0)
print(f"S = {cholesterol.std(ddof=1):.2f} mg/dL (σ assumed before: {SIGMA:.0f});  t = {t1:.4f}, df = {df1}, p = {p1:.4f}")
print("scipy.stats.ttest_1samp:", stats.ttest_1samp(cholesterol, MU0))
print(f"(z-test with known σ gave z = {z_obs:.4f}, p = {p_obs:.4f})")

🔍 **What just happened?** The sample SD is 38.11 mg/dL, a little below the assumed 40, so $|t| = 2.343$ is slightly
larger than $|z| = 2.233$; with 99 degrees of freedom the $t$ distribution is nearly normal, and $p = 0.021$ is close to
the $z$-test's 0.026. Our function and `scipy.stats.ttest_1samp` agree exactly. For large samples the choice between $z$
and $t$ hardly matters; for small samples it matters a lot (Section 7).

### Paired data

Many times the data we have are a set of samples measured **in two different conditions** — for example, a set of
patients measured **before and after** treatment. We want to know whether the treatment made a consistent change across
the population. We also do not know where each individual sits compared with some (unknown) population mean μ: patients
differ a lot from each other.

The hypotheses are $H_0: X_1 = X_2$ vs $H_1: X_1\ne X_2$, where $X_1$ and $X_2$ are the measurements under the two
experimental conditions. Another way to say this is to define the **within-patient difference** $\delta = X_1-X_2$ and test

$$H_0:\ \mu_\delta = 0\qquad\text{vs}\qquad H_1:\ \mu_\delta\ne0 ,$$

*In words:* on average over patients, is the before-minus-after difference zero? (Writing "$\delta = 0$" is shorthand for
"the mean difference is 0".)

As before we compute the mean $\overline{\delta_n}$ and the standard deviation $S_\delta$ of the $n$ differences, and use the
statistic

$$T = \frac{\overline{\delta_n}-0}{S_\delta/\sqrt n}\ \sim\ t_{n-1}\quad\text{under }H_0 .$$

*In words:* a paired $t$-test is just a one-sample $t$-test on the differences, with null value 0.

⚠️ **Common confusion.** You may see the paired statistic written with "$\overline{\delta_n}-\mu_0$" in the numerator.
Under $H_0:\mu_\delta = 0$ the subtracted value is **0** (more generally, the hypothesised mean *difference* $\delta_0$) —
not the population mean of the measurements, which pairing removes and we never need to know.

In [ ]:
# ── 8.2 The paired trial: 12 patients before and after the drug (SIMULATED, simulate_cholesterol_trial) ──
trial = simulate_cholesterol_trial()          # LDL cholesterol (mg/dL); true effect −18 mg/dL (hidden)
trial["difference (before − after)"] = trial["before"] - trial["after"]
print(trial.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})
for _, r in trial.iterrows():
    axes[0].plot([0, 1], [r["before"], r["after"]], "-o", color="tab:blue", alpha=0.6, ms=4)
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["before", "after"]); axes[0].set_xlim(-0.3, 1.3)
axes[0].set_ylabel("LDL cholesterol (mg/dL)"); axes[0].set_title("each line = one patient")
d = trial["difference (before − after)"]
axes[1].scatter(np.zeros(len(d)) + rng.uniform(-0.05, 0.05, len(d)), d, color="tab:purple")
axes[1].axhline(0, color="k", lw=0.8); axes[1].axhline(d.mean(), color="tab:purple", ls="--", label=f"mean = {d.mean():.2f}")
axes[1].set_xticks([]); axes[1].set_xlim(-0.5, 0.5); axes[1].set_ylabel("before − after (mg/dL)")
axes[1].set_title("the 12 differences δ"); axes[1].legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The left panel ("spaghetti plot") shows why pairing matters: patients start anywhere between
about 105 and 196 mg/dL — a huge spread — but most lines slope **down**. The right panel keeps only what we care about,
the 12 within-patient differences: 9 of 12 are positive (cholesterol fell), and their mean is **12.875 mg/dL**. The
patient-to-patient spread has disappeared from this view.

In [ ]:
# ── 8.3 The paired t-test by hand, with SciPy, and with the t table ──
def paired_t(before, after):
    """Paired t-test of H0: mean(before − after) = 0 (two-sided). Returns (mean diff, SD of diffs, t, df, p)."""
    dlt = np.asarray(before, dtype=float) - np.asarray(after, dtype=float)
    n = dlt.size
    t = (dlt.mean() - 0.0) / (dlt.std(ddof=1) / np.sqrt(n))
    return dlt.mean(), dlt.std(ddof=1), t, n - 1, 2 * stats.t.sf(abs(t), n - 1)


dbar, sd_d, t_p, df_p, p_p = paired_t(trial["before"], trial["after"])
print(f"δ̄ = {dbar:.3f} mg/dL, S_δ = {sd_d:.3f}, S_δ/√n = {sd_d / np.sqrt(12):.3f}")
print(f"t = {dbar:.3f} / {sd_d / np.sqrt(12):.3f} = {t_p:.4f}, df = {df_p}, two-sided p = {p_p:.4f}")
print("scipy.stats.ttest_rel:", stats.ttest_rel(trial["before"], trial["after"]))
row11 = t_table.loc[11]
print(f"t table, row ν = 11: {row11['0.005']:.3f} (0.005) < t = {t_p:.3f} < {row11['0.001']:.3f} (0.001) "
      f"→ one-tailed p between 0.001 and 0.005, two-sided between 0.002 and 0.01")

🔍 **What just happened?** The mean difference is 12.875 mg/dL with SD 12.79, so the standard error is 3.693 and
**t = 3.486** on 11 degrees of freedom: **p = 0.0051**, very strong evidence (by the table of Section 6: "< 0.01") that the
drug lowers LDL. `scipy.stats.ttest_rel` gives the identical statistic and $p$-value, and the table bracket
(3.106 < 3.486 < 4.025 in row 11) agrees. The simulation's hidden truth was a −18 mg/dL effect, so the test made the
right call.

**What if we had ignored the pairing?** An *unpaired* two-sample test (Section 9) treats the "before" and "after" columns
as two unrelated groups of patients.

In [ ]:
# ── 8.4 Same data, unpaired analysis — and why it loses ──
welch = stats.ttest_ind(trial["before"], trial["after"], equal_var=False)
print(f"unpaired (Welch) test on the same 24 numbers: t = {welch.statistic:.4f}, p = {welch.pvalue:.4f}")
r_ba = np.corrcoef(trial["before"], trial["after"])[0, 1]
v_b, v_a = trial["before"].var(ddof=1), trial["after"].var(ddof=1)
cov_ba = np.cov(trial["before"], trial["after"])[0, 1]
print(f"Var(before) = {v_b:.1f}, Var(after) = {v_a:.1f}, Cov = {cov_ba:.1f}, correlation = {r_ba:.3f}")
print(f"Var(before − after) = Var(b) + Var(a) − 2 Cov = {v_b + v_a - 2 * cov_ba:.1f}  "
      f"(= S_δ² = {sd_d ** 2:.1f});  without the covariance term it would be {v_b + v_a:.1f}")

# power over 1,000 simulated trials of the same design
p_pair, p_unpair = [], []
for s in range(1000):
    tr = simulate_cholesterol_trial(seed=s)
    p_pair.append(stats.ttest_rel(tr["before"], tr["after"]).pvalue)
    p_unpair.append(stats.ttest_ind(tr["before"], tr["after"], equal_var=False).pvalue)
print(f"\npower at α = 0.05 over 1,000 simulated trials: paired {np.mean(np.array(p_pair) < 0.05):.3f}, "
      f"unpaired {np.mean(np.array(p_unpair) < 0.05):.3f}")

🔍 **What just happened?** Analysed as two unrelated groups, the same 24 numbers give $p = 0.25$ — no evidence at all.
The reason is in the variance algebra of Module 21: $\operatorname{Var}(X_1-X_2) = \operatorname{Var}X_1 + \operatorname{Var}X_2
- 2\operatorname{Cov}(X_1,X_2)$. Before and after values of the same patient are strongly correlated ($r = 0.911$), so the
covariance term removes most of the between-patient variance: the variance of the differences is **163.6** instead of
**1,432** — almost nine times smaller. Over 1,000 simulated trials the paired test detects the drug in about **98 %** of
them, the unpaired test in about **11 %**. Same patients, same measurements, a completely different answer: **the
analysis must match the design.**

### Tests and confidence intervals are two views of the same thing

A 95 % confidence interval for the mean difference (Module 22, with the $t$ quantile because σ is estimated) is

$$\overline{\delta_n}\ \pm\ t_{n-1,\,0.975}\,\frac{S_\delta}{\sqrt n}.$$

*In words:* the estimate, plus or minus about two standard errors. **Duality:** the two-sided test at level 0.05 rejects
$H_0:\mu_\delta=\delta_0$ exactly when $\delta_0$ lies **outside** the 95 % confidence interval — the interval is the set of
all null values the data do not reject.

In [ ]:
# ── 8.5 Confidence interval ↔ test: the CI is the set of null values that are NOT rejected ──
se_d = sd_d / np.sqrt(12)
t975 = stats.t.ppf(0.975, 11)
ci = (dbar - t975 * se_d, dbar + t975 * se_d)
print(f"95% CI for the mean drop: {dbar:.3f} ± {t975:.3f} × {se_d:.3f} = [{ci[0]:.2f}, {ci[1]:.2f}] mg/dL")

null_values = np.linspace(-5, 30, 3501)
p_for_null = 2 * stats.t.sf(np.abs((dbar - null_values) / se_d), 11)
kept = null_values[p_for_null > 0.05]
print(f"null values δ0 with p > 0.05 run from {kept.min():.2f} to {kept.max():.2f} — the same interval")
print(f"0 is {'inside' if ci[0] <= 0 <= ci[1] else 'outside'} the CI  ⇔  H0: δ = 0 is "
      f"{'retained' if ci[0] <= 0 <= ci[1] else 'rejected'} (p = {p_p:.4f})")

🔍 **What just happened?** The 95 % CI is **[4.75, 21.00] mg/dL**. Scanning 3,501 candidate null values, exactly those
between 4.75 and 21.00 have $p>0.05$: the interval *is* the set of non-rejected hypotheses. Zero lies outside it, which is
the same statement as "reject $H_0:\delta=0$ at α = 0.05". The CI says more than the test: it gives the plausible *size*
of the effect (somewhere between about 5 and 21 mg/dL).

### Statistical vs practical significance

A tiny $p$-value says the effect is probably not zero — not that it matters. With enough patients, even a clinically
irrelevant effect becomes "highly significant".

In [ ]:
# ── 8.6 A huge trial with a negligible effect (SIMULATED) ──
rng_big = np.random.default_rng(SEED + 86)
n_big = 40_000
diff_big = rng_big.normal(0.5, 12.8, n_big)         # true mean drop: 0.5 mg/dL — clinically irrelevant
t_big = diff_big.mean() / (diff_big.std(ddof=1) / np.sqrt(n_big))
p_big = 2 * stats.t.sf(abs(t_big), n_big - 1)
half = stats.t.ppf(0.975, n_big - 1) * diff_big.std(ddof=1) / np.sqrt(n_big)
print(f"n = {n_big:,}: mean drop = {diff_big.mean():.3f} mg/dL, t = {t_big:.2f}, p = {p_big:.2e}")
print(f"95% CI: [{diff_big.mean() - half:.3f}, {diff_big.mean() + half:.3f}] mg/dL")

🔍 **What just happened?** With 40,000 patients and a true average drop of only 0.5 mg/dL, the test is overwhelmingly
"significant" ($t = 8.2$, $p\approx3\times10^{-16}$) — yet the confidence interval shows the effect is between roughly 0.40
and 0.65 mg/dL,
far too small to matter for any patient. **Statistical significance** answers "is it zero?"; **practical (biological,
clinical) significance** asks "is it big enough to care?". Always report the effect size with its confidence interval,
not just the $p$-value.

### 📘 Lecture notes — paired $t$-tests, confidence intervals and significance

- **Paired data:** the same units measured in two conditions (patients before/after treatment). We want to know whether
  the treatment made a consistent change, without knowing where each individual sits relative to the population mean.
- **Hypotheses:** $H_0: X_1 = X_2$ vs $H_1: X_1\ne X_2$; with $\delta = X_1-X_2$: $H_0:\mu_\delta=0$ vs $H_1:\mu_\delta\ne0$.
- **Statistic:** $T = \dfrac{\overline{\delta_n}-0}{S_\delta/\sqrt n}\sim t_{n-1}$ — a one-sample $t$-test on the differences.
- **Worked example** (12 patients): $\bar\delta = 12.875$, $S_\delta = 12.79$, $t = 3.486$, df = 11, $p = 0.0051$; an
  unpaired test of the same numbers gives $p = 0.25$, because pairing removes the between-patient variance
  ($\operatorname{Var}(X_1-X_2) = \operatorname{Var}X_1+\operatorname{Var}X_2-2\operatorname{Cov}$).
- **Duality:** the level-0.05 two-sided test rejects $\delta_0$ iff $\delta_0\notin$ the 95 % CI
  $\bar\delta\pm t_{n-1,0.975}S_\delta/\sqrt n$ (here [4.75, 21.00]).
- **Significance ≠ importance:** with huge $n$, negligible effects give tiny $p$; report effect sizes and CIs.

> 🎓 **Exam tip.** When the same subjects appear in both conditions, use the paired test; when they are different subjects, use a two-sample test. Using the wrong one is one of the most common errors in biology papers — in either direction.

✏️ **Your turn.** Simulate a trial with `simulate_cholesterol_trial(n=6, effect=-18.0, seed=1)` and run both tests. Then
try `n=30`. How large must the trial be before the *unpaired* test also detects the drug reliably?

---
## 9 · Two independent groups: Welch's $t$-test and a first differential-expression test

> 📍 **Where we are.** Pairing is not always possible: infected and control mice are *different* animals. Comparing two
> independent groups needs a two-sample test — the workhorse of differential gene expression.

Let $x_{1},\dots,x_{n_1}$ (infected) and $y_{1},\dots,y_{n_2}$ (control) be independent samples from normal distributions
with means $\mu_1,\mu_2$ and possibly different variances. To test $H_0:\mu_1=\mu_2$ vs $H_1:\mu_1\ne\mu_2$, **Welch's
$t$-test** uses

$$T = \frac{\bar x-\bar y}{\sqrt{\dfrac{s_1^2}{n_1}+\dfrac{s_2^2}{n_2}}}, \qquad
\nu \approx \frac{\Big(\dfrac{s_1^2}{n_1}+\dfrac{s_2^2}{n_2}\Big)^2}{\dfrac{(s_1^2/n_1)^2}{n_1-1}+\dfrac{(s_2^2/n_2)^2}{n_2-1}} .$$

*In words:* the difference of the two sample means, measured in units of its standard error (the variance of a difference
of *independent* means is the sum of their variances — Module 21, no covariance term now). The degrees of freedom ν
(the Welch–Satterthwaite formula) lie between $\min(n_1,n_2)-1$ and $n_1+n_2-2$; under $H_0$, $T$ is approximately $t_\nu$.

We take the infection experiment of the course: 4 control and 4 infected spleen samples, log2 expression. *MX1* is an
interferon-stimulated antiviral gene; *GAPDH* is a housekeeping gene that should not respond.

In [ ]:
# ── 9.1 Welch's t-test by hand for one gene, infected vs control spleen (SIMULATED) ──
expr = simulate_tissue_expression()
X_expr, genes, samples = expr["X"], expr["genes"], expr["samples"]


def welch_t(a, b):
    """Welch two-sample t-test of H0: mean(a) = mean(b) (two-sided). Returns (t, df, p)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    va, vb = a.var(ddof=1) / a.size, b.var(ddof=1) / b.size
    t = (a.mean() - b.mean()) / np.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (a.size - 1) + vb ** 2 / (b.size - 1))
    return t, df, 2 * stats.t.sf(abs(t), df)


in_spleen = samples["tissue"] == "spleen"
infected = in_spleen & (samples["condition"] == "infected")
control = in_spleen & (samples["condition"] == "control")
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=False)
for ax, gene in zip(axes, ["MX1", "GAPDH"]):
    j = genes.index(gene)
    a, b = X_expr[infected, j], X_expr[control, j]
    t_w, df_w, p_w = welch_t(a, b)
    sp = stats.ttest_ind(a, b, equal_var=False)
    print(f"{gene:6s} infected {np.round(a, 2)}  control {np.round(b, 2)}")
    print(f"       difference = {a.mean() - b.mean():+.3f} log2 units; t = {t_w:.3f}, ν = {df_w:.2f}, p = {p_w:.2e} "
          f"| scipy: t = {sp.statistic:.3f}, p = {sp.pvalue:.2e} | true difference = {expr['true_lfc'].loc['spleen', gene]:.3f}")
    ax.scatter(np.zeros(4), b, color="tab:blue", label="control"); ax.scatter(np.ones(4), a, color="tab:red", label="infected")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["control", "infected"]); ax.set_xlim(-0.5, 1.5)
    centre = np.concatenate([a, b]).mean()
    ax.set_ylim(centre - 2.25, centre + 2.25)                         # same 4.5-unit span in both panels
    ax.set_ylabel("log2 expression"); ax.set_title(f"{gene} in spleen: p = {p_w:.2g}")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** *MX1* is about **2.76 log2 units** higher in infected spleen (roughly 6.8-fold, since
$2^{2.76}\approx6.8$), with $t = 8.58$ on about 5.3 degrees of freedom and $p = 2.6\times10^{-4}$: strong evidence of induction (the simulation's true
difference is 2.40). *GAPDH* differs by only −0.016 log2 units with $p = 0.81$ — no evidence of change, as expected for a
housekeeping gene. Our function matches `scipy.stats.ttest_ind(..., equal_var=False)` exactly. Note the fractional
degrees of freedom: Welch's ν is an approximation and need not be an integer. (Both panels span the same 4.5 log2
units vertically, so the visual difference between the genes is honest.)

A related test, **Student's two-sample $t$-test**, assumes the two groups have equal variances and pools them
($\nu = n_1+n_2-2$); [Module 27 · 📉 Linear regression: the whole course in one model](27_linear_regression_review.ipynb) shows that it is exactly a linear regression on a 0/1 "infected" indicator.

### 🌉 Bridge — From one gene to a whole genome

A transcriptomics experiment does not test one gene; it tests **all** of them. The infection data set has 400 genes in
three tissues; a real RNA-seq study has about 20,000. Run Welch's test on every gene at α = 0.05 and, even if nothing
changed, about 5 % of the genes — 1,000 of 20,000 — would come out "significant". [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb) develops the tools that make
genome-wide testing honest — the expected number of false positives, Bonferroni, the false discovery rate and the
Benjamini–Hochberg procedure — and applies them to this very data set with volcano plots and Venn diagrams of the
discovered genes. Real pipelines replace the $t$-test by count models (negative-binomial GLMs, as in DESeq2 and edgeR)
and borrow information across genes to stabilise the variance estimates of small experiments — see
[BiologyToML Module 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb).

✏️ **Your turn.** Repeat the test for *MX1* in the kidney and the liver, and for the cytokine *IL6* in all three
tissues. Then pick one gene named `g…` at random (e.g. `"g101"`): is it significant? Check against
`expr["true_lfc"]` whether it truly changed.

## 📌 Key takeaways

- **Testing asks a different question from estimation:** not "what is μ?" but "are the data compatible with a specific
  claim, $H_0$?". The decision is *reject* or *retain* $H_0$ — never "accept".
- **Hypotheses partition the parameter space:** $H_0:\theta\in\Theta_0$, $H_1:\theta\in\Theta_1$; two-sided
  ($\theta = \theta_0$ vs $\ne$) or one-sided ($\le$ vs $>$, $\ge$ vs $<$), fixed before seeing the data.
- **Two kinds of error:** Type I (false positive, rate $\alpha = P(\text{reject}\mid H_0)$) and Type II (false negative,
  rate $\beta = P(\text{retain}\mid H_1)$); **power** $=1-\beta$.
- **The two-bell-curve picture:** α is the $H_0$ area beyond the cut-off(s), β the $H_1$ area inside. Moving the cut-off
  trades one for the other; a larger effect $\Delta$, a smaller σ or a larger $n$ improves both — the separation is
  $\sqrt n\,\Delta/\sigma$, and $n = \big((z_{1-\alpha/2}+z_{1-\beta})\sigma/\Delta\big)^2$ (126 patients for our drug).
- **Three steps:** statistic → $p$-value (probability *under $H_0$* of something at least as extreme) → decision
  ($p\le\alpha$). Under $H_0$, $p$-values are uniform; a $p$-value is not $P(H_0)$.
- **$z$ vs $t$:** with known σ, $Z = (\bar X-\mu_0)/(\sigma/\sqrt n)\sim\mathcal N(0,1)$; with estimated σ,
  $T = (\bar X-\mu_0)/(S/\sqrt n)\sim t_{n-1}$ — heavier tails, larger cut-offs, approaching the normal as $n$ grows.
- **Match the analysis to the design:** paired data → paired $t$-test on the differences (our trial: $p=0.005$ paired vs
  $0.25$ unpaired); independent groups → Welch's two-sample test.
- **Report effects with confidence intervals:** the CI is the set of null values the test does not reject, and it
  separates statistical from practical significance.

---
## ✅ Checkpoint

1. A study tests $H_0:\mu = 120$ mmHg (mean systolic blood pressure) against $H_1:\mu\ne120$. Describe a Type I and a Type
   II error in words.
   <details><summary>Answer</summary>Type I: the test concludes that mean blood pressure differs from 120 when it really is 120 (false positive). Type II: the test retains $H_0$ although the true mean differs from 120 (false negative — a real difference is missed).</details>
2. By hand: $n = 25$ measurements, known $\sigma = 10$, $\bar x = 104$, $H_0:\mu=100$, two-sided. Compute $z$ and $p$, and
   decide at α = 0.05.
   <details><summary>Answer</summary>Standard error $10/\sqrt{25} = 2$, so $z = (104-100)/2 = 2$. $p = 2(1-\Phi(2)) = 2\times0.0228 = 0.0455 < 0.05$ → reject $H_0$ (strong evidence, by the evidence table). Check: `2*stats.norm.sf(2)`.</details>
3. For the test of question 2, what is the power against a true mean of 105? (Use the cut-offs $100\pm1.96\cdot2$.)
   <details><summary>Answer</summary>Retain region $[96.08, 103.92]$. Under $\mu^*=105$: $\beta = \Phi\big(\frac{103.92-105}{2}\big)-\Phi\big(\frac{96.08-105}{2}\big) = \Phi(-0.54)-\Phi(-4.46)\approx0.295$, power ≈ 0.705 — the same as our cholesterol study, because the separation $\sqrt n\Delta/\sigma = 5\cdot5/10 = 2.5$ is the same.</details>
4. True or false, with a reason: "If $\mu^*$ moves further from $\mu_0$, both the Type I and Type II error rates decrease."
   <details><summary>Answer</summary>False as stated. With a fixed cut-off, α depends only on the null distribution and the cut-off, so it does not change; β decreases (power increases). A larger separation does let you choose a stricter cut-off and still keep β small, so the *achievable* pair improves.</details>
5. How many patients are needed to detect a 5 mg/dL change in mean cholesterol with σ = 40, two-sided α = 0.05, power
   0.9? ($z_{0.9} = 1.2816$.)
   <details><summary>Answer</summary>$n = \big((1.96+1.2816)\cdot40/5\big)^2 = (25.93)^2 \approx 672.5$ → 673 patients.</details>
6. Explain why $P(p\le\alpha\mid H_0) = \alpha$ for a continuous test statistic.
   <details><summary>Answer</summary>The $p$-value is $1-F(T)$ (one-sided case) where $F$ is the null CDF of the statistic $T$. For a continuous $T$, $F(T)$ is Uniform(0,1) under $H_0$ (the probability-integral transform), so $p$ is uniform too and $P(p\le\alpha)=\alpha$. Equivalently, "$p\le\alpha$" is the event "$T$ beyond the α cut-off", which has probability α under $H_0$ by construction.</details>
7. A one-sample $t$-test with $n = 9$ gives $t = 2.5$. Bracket the two-sided $p$-value using the $t$ table.
   <details><summary>Answer</summary>ν = 8. Row 8: 2.306 (0.025) < 2.5 < 2.896 (0.01). One-tailed $p$ between 0.01 and 0.025, two-sided between 0.02 and 0.05. Exact: `2*stats.t.sf(2.5, 8)` = 0.037.</details>
8. Why does the $t$ distribution have heavier tails than the normal, and why does the difference vanish as $n$ grows?
   <details><summary>Answer</summary>Because the denominator $S$ is estimated and random: when $S$ happens to be small, $T$ is inflated, which produces extra extreme values. As $n\to\infty$, $S\to\sigma$ (law of large numbers), so $T$ behaves like $Z$ and $t_{n-1}\to\mathcal N(0,1)$.</details>
9. Ten mice are weighed before and after a diet; the differences have mean 1.2 g and SD 1.5 g. Perform the paired test.
   <details><summary>Answer</summary>$t = 1.2/(1.5/\sqrt{10}) = 1.2/0.474 = 2.53$, ν = 9. Table row 9: 2.262 (0.025) < 2.53 < 2.821 (0.01) → two-sided $p$ between 0.02 and 0.05 (exactly 0.032): reject $H_0$ at 0.05.</details>
10. In our cholesterol trial the paired test gave $p=0.005$ and the unpaired test $p=0.25$. Which variance does pairing
    remove, and which formula shows it?
    <details><summary>Answer</summary>The between-patient variance (patients' baseline levels differ by SD ≈ 30 mg/dL). $\operatorname{Var}(X_1-X_2) = \operatorname{Var}X_1+\operatorname{Var}X_2-2\operatorname{Cov}(X_1,X_2)$; the strong positive covariance of a patient's before and after values cancels most of the variance (163.6 vs 1,432 here).</details>
11. A 95 % CI for a mean difference is [−1.2, 6.8]. What is the decision of the two-sided test of $H_0:\mu_\delta=0$ at
    α = 0.05? Of $H_0:\mu_\delta=7$?
    <details><summary>Answer</summary>0 is inside the CI → retain $H_0:\mu_\delta=0$ ($p>0.05$). 7 is outside → reject $H_0:\mu_\delta = 7$ ($p<0.05$).</details>
12. Only 5 % of screened compounds are truly active; each is tested at α = 0.05 with power 0.8. What fraction of "hits"
    are false?
    <details><summary>Answer</summary>$P(H_0\mid\text{reject}) = \frac{0.05\cdot0.95}{0.05\cdot0.95+0.8\cdot0.05} = \frac{0.0475}{0.0875}\approx0.54$ — more than half of the hits are false positives. This is the false discovery rate of Module 26.</details>

## 🧠 Questions & discussion

1. A paper reports "no significant effect of the knockout ($p = 0.3$, $n = 3$ mice per group)" and concludes that the gene
   is not needed for the phenotype. What would you ask the authors? What analysis would convince you?
2. When is a one-sided test scientifically justified? Think of a drug that could plausibly be harmful as well as helpful.
3. The threshold 0.05 is a convention. Discuss the costs of moving it to 0.005 for (a) an exploratory screen of thousands
   of genes and (b) a confirmatory clinical trial.
4. Our $t$-tests assume normal data. Which biological measurements are clearly not normal (think of counts, ratios,
   survival times), and what could you do — transform, use another test, or rely on the CLT?
5. Why might biological replicates (different mice) and technical replicates (the same RNA sequenced twice) give
   very different $p$-values for the same comparison? Which one answers the biological question?

## 🚀 Bonus — going further

- **Non-parametric tests.** Implement the sign test for the paired trial (count positive differences; under $H_0$ the
  count is Binomial(12, ½)) and compare with `stats.wilcoxon` and with the paired $t$-test.
- **A permutation test.** For *MX1* in spleen, shuffle the 8 condition labels in all $\binom84 = 70$ ways, recompute the
  mean difference each time, and use the fraction at least as extreme as observed as an exact $p$-value.
- **Power of the $t$-test.** Use `stats.nct` (the non-central $t$ distribution) to compute the exact power of the paired
  test for $n = 6$ and compare with a simulation.
- **Likelihood-ratio tests.** Read how Neyman and Pearson (1933) showed that, for simple hypotheses, rejecting when the
  likelihood ratio is large gives the most powerful test (the Neyman–Pearson lemma, Wasserman §10.6).
- **The ASA statement on $p$-values** (Wasserstein & Lazar 2016) — six principles in two pages; worth reading in full.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 25 — Hypothesis testing: errors, power, z- and t-tests            Date:
Input: SIMULATED cholesterol after medication (100 patients, mu0 = 210, sigma = 40, true mu = 200);
       SIMULATED paired trial (simulate_cholesterol_trial, 12 patients); SIMULATED spleen expression (4 vs 4)
Output: z, t, p-values, decisions; alpha/beta/power; required sample sizes; confidence intervals
Method: three-step test (statistic, p-value under H0, decision at alpha); z-test, one-sample/paired/Welch t-tests
Key equation: Z = (xbar − mu0)/(sigma/√n) ~ N(0,1);  T = (xbar − mu0)/(S/√n) ~ t_(n−1);
              n = ((z_(1−alpha/2) + z_(1−beta)) sigma / Delta)²
Key parameters: alpha, one- vs two-sided, n, sigma, effect size Delta
What changed when I modified …: (e.g. TRUE_MU, the cut-off, n, pairing vs not)
One failure mode: (e.g. normal cut-offs with an estimated sigma in small samples; unpaired test on paired data)
Connection to 02-680: hypothesis testing — errors, power, z- and t-tests, paired t-tests
```

## 📚 References

- Wasserman L. (2004) *All of Statistics: A Concise Course in Statistical Inference*. Springer — Chapter 10 (hypothesis
  testing and $p$-values).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — Chapter 9 (testing hypotheses).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences*. CRC Press.
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Student (1908) The probable error of a mean. *Biometrika* 6:1–25.
- Neyman J., Pearson E. S. (1933) On the problem of the most efficient tests of statistical hypotheses.
  *Philosophical Transactions of the Royal Society A* 231:289–337.
- Welch B. L. (1947) The generalization of "Student's" problem when several different population variances are
  involved. *Biometrika* 34:28–35.
- Button K. S. *et al.* (2013) Power failure: why small sample size undermines the reliability of neuroscience.
  *Nature Reviews Neuroscience* 14:365–376.
- Wasserstein R. L., Lazar N. A. (2016) The ASA statement on p-values: context, process, and purpose.
  *The American Statistician* 70:129–133.

## ➡️ Next

[Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb) carries these ideas to data that come in **categories** — Mendel's pea counts, genotype frequencies,
genotype × disease tables — with the χ² test, and then to the reality of genomics: thousands of tests at once. The
uniform null distribution of $p$-values (Section 6) and the Bayes-rule calculation of false hits (Section 6 🌉) are
exactly what the false discovery rate and the Benjamini–Hochberg procedure are built on.